# RPM and SimCLR on CIFAR-10 — self-contained notebook

This notebook is a **fully self-contained** port of a private codebase: it does
not `pip install` or `import` anything from that repo or from GitHub. Every
piece of code needed to run the two experiments — the Gaussian RPM
(Recognition-Parametrised Model) and SimCLR — is defined inline below, in the
same dependency order as the original modules:

1. `dists` — Gaussian / diagonal-Gaussian / Categorical natural-parameter
   distributions and their KL divergences.
2. `utils` — small initializers.
3. `networks` — recognition-network backbones (MLP / CNN / ResNet / ViT), each
   producing a `(output, aux)` pair where `aux["trunk"]` / `aux["projection"]`
   are the pre/post-projection representations.
4. `rpms` — the Gaussian and Categorical RPM models and their free-energy
   objectives.
5. `simclr` — a SimCLR encoder that reuses the same network backbones (shared
   weights across views, no per-factor `nn.vmap`) plus the NT-Xent loss.
6. CIFAR-10 data loading (multi-view SimCLR-style augmentation).
7. Linear-probe evaluation utilities.
8. The generic training loop (works for both RPM and SimCLR — the loss
   function is dispatched by `model.rpm_type`).

Then two runnable experiments: **Experiment 1** trains a Gaussian RPM,
**Experiment 2** trains SimCLR, both with a ResNet-18-style backbone on
CIFAR-10 and periodic linear-probe evaluation.

**Note on fidelity:** this is a faithful port of the working code, with three
small correctness fixes applied along the way (each is called out inline with
a `# FIX:` comment where it happens):
- `create_feature_extractor`'s `"latent"` branch used to read `.mean` off a
  natural-parameter object that doesn't have that attribute — fixed to read
  the separately-computed mean-parameter object instead.
- Its `"concatenated"` branch used to discard the tuple element that actually
  holds the latent mean — fixed to keep it.
- The best-checkpoint selection compared free energy the wrong way (it's
  *maximized* during training — verified empirically: free energy climbs over
  training — but the checkpoint logic was tracking a *minimum*). Fixed to
  track the maximum.

Everything else is preserved as-is, bugs and all, per request.


## Setup


In [ ]:
# !pip install -q "jax[cpu]" flax optax orbax-checkpoint einops 2>/dev/null
# # If you have a GPU runtime in Colab (Runtime > Change runtime type > GPU),
# # install the CUDA build of jax instead for a large speedup, e.g.:
# #   !pip install -q -U "jax[cuda12]"
# print("Install complete.")


In [ ]:
import math
import os
import sys
import random
import time
from pathlib import Path
from typing import Any, Iterable, Iterator, Mapping, Optional, Sequence, Tuple

# import einops
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import orbax.checkpoint as ocp


from flax import linen as nn
from flax import serialization, struct
from flax.typing import VariableDict
from jax import Array, jit, value_and_grad, vmap
from jax.lax import batch_matmul, fori_loop
from jax.nn import log_softmax, softmax
from jax.scipy.special import logsumexp

print("JAX devices:", jax.devices())


## Part 1 — Distributions (`dists`)

Natural-parameter containers for full-covariance Gaussian, diagonal Gaussian,
and Categorical distributions, plus their KL divergences and log-normalizers.


In [ ]:
@struct.dataclass
class GaussianMeanParams:
    """Mean parameters of a full-covariance Gaussian."""

    mean: Array
    cov: Array


@struct.dataclass
class DiagGaussianMeanParams:
    """Mean parameters of a diagonal Gaussian."""

    mean: Array
    var_diag: Array


@jit
def mvn_lognormalizer(precision: Array, pwm: Array, mean: Array | None = None) -> Array:
    """Log-partition of a Gaussian given precision and precision-weighted mean.

    If ``mean`` (= precision^{-1} @ pwm) is already known it can be passed to
    avoid re-solving the linear system.
    """
    d = pwm.shape[-1]

    if mean is None:
        mean = jnp.linalg.solve(precision, pwm[..., None])[..., 0]

    offset = d * jnp.log(2 * jnp.pi)
    dets = jnp.linalg.slogdet(precision)[1]
    quad = jnp.sum(pwm * mean, axis=-1)
    lognorm = 0.5 * (offset - dets + quad)

    return lognorm


@jit
def mvn_log_prob(
    x: Array, mean_params: GaussianMeanParams | DiagGaussianMeanParams
) -> Array:
    """Log-probability of a Gaussian given mean parameters (mean and covariance)."""
    mean = mean_params["mean"]
    if "cov" in mean_params:
        cov = mean_params["cov"]
        d = mean.shape[-1]
        offset = d * jnp.log(2 * jnp.pi)
        dets = jnp.linalg.slogdet(cov)[1]
        quad = jnp.sum(
            (x - mean)
            * batch_matmul(jnp.linalg.inv(cov), (x - mean)[..., None])[..., 0],
            axis=-1,
        )
        return -0.5 * (offset + dets + quad)
    elif "var_diag" in mean_params:
        var_diag = mean_params["var_diag"]
        d = mean.shape[-1]
        offset = d * jnp.log(2 * jnp.pi)
        log_det = jnp.sum(jnp.log(var_diag), axis=-1)
        quad = jnp.sum(((x - mean) ** 2) / var_diag, axis=-1)
        return -0.5 * (offset + log_det + quad)
    else:
        raise ValueError("mean_params must contain either 'cov' or 'var_diag'.")


@struct.dataclass
class GaussianNatParams:
    """Batched natural parameters of a full-covariance Gaussian.

    Supports slicing and broadcasting arithmetic (sum / subtract / scale).
    """

    precision: Array
    precision_weighted_mean: Array

    def event_shape(self) -> int:
        return self.precision_weighted_mean.shape[-1]

    def batch_shape(self) -> tuple[int, ...]:
        return self.precision_weighted_mean.shape[:-1]

    def sum(self, batch_axis: int) -> "GaussianNatParams":
        # Only leading (batch) axes may be summed; the trailing axis is the
        # event dimension.
        if batch_axis > len(self.precision_weighted_mean.shape) - 2:
            raise ValueError(
                "Can only sum over batch axes, batch shape is: "
                f"{self.precision_weighted_mean.shape[:-1]} got axis {batch_axis}"
            )

        return GaussianNatParams(
            self.precision.sum(batch_axis), self.precision_weighted_mean.sum(batch_axis)
        )

    def expand_batch_dim(self, axis: int) -> "GaussianNatParams":
        return GaussianNatParams(
            jnp.expand_dims(self.precision, axis),
            jnp.expand_dims(self.precision_weighted_mean, axis),
        )

    def __mul__(self, alpha: float) -> "GaussianNatParams":
        return GaussianNatParams(
            alpha * self.precision, alpha * self.precision_weighted_mean
        )

    def __rmul__(self, alpha: float) -> "GaussianNatParams":
        return self.__mul__(alpha)

    def __add__(self, other: "GaussianNatParams") -> "GaussianNatParams":
        shape_a, shape_b = self.precision.shape, other.precision.shape
        try:
            jnp.broadcast_shapes(shape_a, shape_b)
        except ValueError:
            raise ValueError(
                f"Shapes must be broadcastable, got {shape_a} and {shape_b}"
            )

        return GaussianNatParams(
            self.precision + other.precision,
            self.precision_weighted_mean + other.precision_weighted_mean,
        )

    def __sub__(self, other: "GaussianNatParams") -> "GaussianNatParams":
        shape_a, shape_b = self.precision.shape, other.precision.shape
        try:
            jnp.broadcast_shapes(shape_a, shape_b)
        except ValueError:
            raise ValueError(
                f"Shapes must be broadcastable, got {shape_a} and {shape_b}"
            ) from None

        return GaussianNatParams(
            self.precision - other.precision,
            self.precision_weighted_mean - other.precision_weighted_mean,
        )

    def __getitem__(self, s) -> "GaussianNatParams":
        return GaussianNatParams(self.precision[s], self.precision_weighted_mean[s])

    def lognormalizer(self, mean: Array | None = None) -> Array:
        return mvn_lognormalizer(self.precision, self.precision_weighted_mean, mean)

    @classmethod
    def from_mean_param(cls, mean_params: GaussianMeanParams) -> "GaussianNatParams":
        precision = jnp.linalg.inv(mean_params.cov)
        precision_weighted_mean = batch_matmul(
            precision, jnp.expand_dims(mean_params.mean, -1)
        ).squeeze()
        return GaussianNatParams(precision, precision_weighted_mean)


@jit
def list_to_batch(dists: Sequence[GaussianNatParams]) -> GaussianNatParams:
    """Stack a sequence of identically-shaped Gaussians along a new axis 0."""
    J = len(dists)
    shape = dists[0].batch_shape()
    new_shape = (J,) + shape
    event_shape = dists[0].event_shape()

    p = jnp.zeros(new_shape + (event_shape, event_shape))
    pwm = jnp.zeros(new_shape + (event_shape,))
    for j in range(J):
        p = p.at[j].set(dists[j].precision)
        pwm = pwm.at[j].set(dists[j].precision_weighted_mean)

    return GaussianNatParams(precision=p, precision_weighted_mean=pwm)


@jit
def mean_params(d: GaussianNatParams) -> GaussianMeanParams:
    """Convert natural params to mean params (mean and covariance)."""
    cov = jnp.linalg.inv(d.precision)
    mean = batch_matmul(cov, jnp.expand_dims(d.precision_weighted_mean, -1)).squeeze(-1)
    return GaussianMeanParams(mean=mean, cov=cov)


@jit
def unnormalized_kl(
    q_mean: Array,
    q_cov: Array,
    q_nat: GaussianNatParams,
    f_nat: GaussianNatParams,
    h_nat: GaussianNatParams | None = None,
    f_phi: Array | None = None,
    h_phi: Array | None = None,
    q_phi: Array | None = None,
) -> Array:
    """Pseudo-KL from ``q`` to the unnormalized density ``f * h``.

    Log-normalizers are recomputed only when not supplied by the caller.
    """
    if f_phi is None:
        f_phi = mvn_lognormalizer(f_nat.precision, f_nat.precision_weighted_mean)

    if q_phi is None:
        q_phi = mvn_lognormalizer(
            q_nat.precision, q_nat.precision_weighted_mean, q_mean
        )

    # E_q[z z^T] = Cov + mean mean^T.
    # q_square = q_cov + einops.einsum(q_mean, q_mean, "... d1, ... d2 -> ... d1 d2")
    q_square = q_cov + jn.einsum(q_mean, q_mean, "... d1, ... d2 -> ... d1 d2")

    if h_nat is None:
        normalizers = q_phi - f_phi
        nat_diff = f_nat - q_nat
    else:
        if h_phi is None:
            h_phi = mvn_lognormalizer(h_nat.precision, h_nat.precision_weighted_mean)
        normalizers = q_phi - f_phi - h_phi
        nat_diff = f_nat + h_nat - q_nat

    linear = jnp.sum(nat_diff.precision_weighted_mean * q_mean, axis=-1)
    quadratic = -0.5 * jnp.sum(nat_diff.precision * q_square, axis=(-1, -2))
    return -1.0 * (linear + quadratic + normalizers)


@jit
def kl(
    q_mean: Array,
    q_cov: Array,
    q_nat: GaussianNatParams,
    p_nat: GaussianNatParams,
    q_phi: Array | None = None,
    p_phi: Array | None = None,
) -> Array:
    """KL divergence ``KL(q || p)`` between two full Gaussians."""
    if q_phi is None:
        q_phi = mvn_lognormalizer(
            q_nat.precision, q_nat.precision_weighted_mean, q_mean
        )
    if p_phi is None:
        p_phi = mvn_lognormalizer(p_nat.precision, p_nat.precision_weighted_mean)

    # q_square = q_cov + einops.einsum(q_mean, q_mean, "... d1, ... d2 -> ... d1 d2")
    q_square = q_cov + jnp.einsum(q_mean, q_mean, "... d1, ... d2 -> ... d1 d2")

    normalizers = q_phi - p_phi
    nat_diff = p_nat - q_nat

    linear = jnp.sum(nat_diff.precision_weighted_mean * q_mean, axis=-1)
    quadratic = -0.5 * jnp.sum(nat_diff.precision * q_square, axis=(-1, -2))

    return -1.0 * (linear + quadratic + normalizers)


# --- Diagonal Gaussian ---


@struct.dataclass
class DiagGaussianNatParams:
    """Gaussian natural parameters with a diagonal precision."""

    precision_diag: Array
    precision_weighted_mean: Array

    def event_shape(self) -> int:
        return self.precision_weighted_mean.shape[-1]

    def batch_shape(self) -> tuple[int, ...]:
        return self.precision_weighted_mean.shape[:-1]

    def sum(self, batch_axis: int) -> "DiagGaussianNatParams":
        if batch_axis > len(self.precision_weighted_mean.shape) - 2:
            raise ValueError(
                "Can only sum over batch axes, batch shape is: "
                f"{self.precision_weighted_mean.shape[:-1]} got axis {batch_axis}"
            )

        return DiagGaussianNatParams(
            self.precision_diag.sum(batch_axis),
            self.precision_weighted_mean.sum(batch_axis),
        )

    def expand_batch_dim(self, axis: int) -> "DiagGaussianNatParams":
        return DiagGaussianNatParams(
            jnp.expand_dims(self.precision_diag, axis),
            jnp.expand_dims(self.precision_weighted_mean, axis),
        )

    def __mul__(self, alpha: float) -> "DiagGaussianNatParams":
        return DiagGaussianNatParams(
            alpha * self.precision_diag, alpha * self.precision_weighted_mean
        )

    def __rmul__(self, alpha: float) -> "DiagGaussianNatParams":
        return self.__mul__(alpha)

    def __add__(self, other: "DiagGaussianNatParams") -> "DiagGaussianNatParams":
        return DiagGaussianNatParams(
            self.precision_diag + other.precision_diag,
            self.precision_weighted_mean + other.precision_weighted_mean,
        )

    def __sub__(self, other: "DiagGaussianNatParams") -> "DiagGaussianNatParams":
        return DiagGaussianNatParams(
            self.precision_diag - other.precision_diag,
            self.precision_weighted_mean - other.precision_weighted_mean,
        )

    def __getitem__(self, s) -> "DiagGaussianNatParams":
        return DiagGaussianNatParams(
            self.precision_diag[s], self.precision_weighted_mean[s]
        )

    def lognormalizer(self, mean: Array | None = None) -> Array:
        return diag_mvn_lognormalizer(
            self.precision_diag, self.precision_weighted_mean, mean
        )

    @classmethod
    def from_mean_param(
        cls, mean_params: DiagGaussianMeanParams
    ) -> "DiagGaussianNatParams":
        precision_diag = 1.0 / mean_params.var_diag
        precision_weighted_mean = mean_params.mean * precision_diag
        return DiagGaussianNatParams(precision_diag, precision_weighted_mean)


@jit
def diag_mvn_lognormalizer(
    precision_diag: Array, pwm: Array, mean: Array | None = None
) -> Array:
    """Log-partition of a diagonal Gaussian (the diagonal analogue of
    :func:`mvn_lognormalizer`)."""
    d = pwm.shape[-1]
    if mean is None:
        mean = pwm / precision_diag
    offset = d * jnp.log(2 * jnp.pi)
    log_det = jnp.sum(jnp.log(precision_diag), axis=-1)
    quad = jnp.sum(pwm * mean, axis=-1)
    lognorm = 0.5 * (offset - log_det + quad)
    return lognorm


@jit
def diag_mean_params(d: DiagGaussianNatParams) -> DiagGaussianMeanParams:
    """Convert diagonal natural params to mean params (mean and variances)."""
    var_diag = 1.0 / d.precision_diag
    mean = d.precision_weighted_mean * var_diag
    return DiagGaussianMeanParams(mean, var_diag)


@jit
def diag_unnormalized_kl(
    q_mean: Array,
    q_var_diag: Array,
    q_nat: DiagGaussianNatParams,
    f_nat: DiagGaussianNatParams,
    h_nat: DiagGaussianNatParams | None = None,
    f_phi: Array | None = None,
    h_phi: Array | None = None,
    q_phi: Array | None = None,
) -> Array:
    """Diagonal-Gaussian analogue of :func:`unnormalized_kl`."""
    f_phi = (
        f_phi
        if f_phi is not None
        else diag_mvn_lognormalizer(f_nat.precision_diag, f_nat.precision_weighted_mean)
    )
    q_phi = (
        q_phi
        if q_phi is not None
        else diag_mvn_lognormalizer(
            q_nat.precision_diag, q_nat.precision_weighted_mean, q_mean
        )
    )

    # E_q[z^2] = Var + mean^2 (per coordinate).
    q_square_diag = q_var_diag + q_mean**2

    if h_nat is None:
        normalizers = q_phi - f_phi
        nat_diff = f_nat - q_nat
    else:
        h_phi = (
            h_phi
            if h_phi is not None
            else diag_mvn_lognormalizer(
                h_nat.precision_diag, h_nat.precision_weighted_mean
            )
        )
        normalizers = q_phi - f_phi - h_phi
        nat_diff = f_nat + h_nat - q_nat

    linear = jnp.sum(nat_diff.precision_weighted_mean * q_mean, axis=-1)
    quadratic = -0.5 * jnp.sum(nat_diff.precision_diag * q_square_diag, axis=-1)

    return -1.0 * (linear + quadratic + normalizers)


@jit
def diag_kl(
    q_mean: Array,
    q_var_diag: Array,
    q_nat: DiagGaussianNatParams,
    p_nat: DiagGaussianNatParams,
    q_phi: Array | None = None,
    p_phi: Array | None = None,
) -> Array:
    """Diagonal-Gaussian analogue of :func:`kl` (``KL(q || p)``)."""
    q_phi = (
        q_phi
        if q_phi is not None
        else diag_mvn_lognormalizer(
            q_nat.precision_diag, q_nat.precision_weighted_mean, q_mean
        )
    )
    p_phi = (
        p_phi
        if p_phi is not None
        else diag_mvn_lognormalizer(p_nat.precision_diag, p_nat.precision_weighted_mean)
    )

    q_square_diag = q_var_diag + q_mean**2

    normalizers = q_phi - p_phi
    nat_diff = p_nat - q_nat

    linear = jnp.sum(nat_diff.precision_weighted_mean * q_mean, axis=-1)
    quadratic = -0.5 * jnp.sum(nat_diff.precision_diag * q_square_diag, axis=-1)

    return -1.0 * (linear + quadratic + normalizers)


# --- Categorical ---


@struct.dataclass
class CategoricalNatParams:
    """Natural parameters (logits) for a Categorical distribution."""

    logits: Array

    def event_shape(self) -> int:
        return self.logits.shape[-1]

    def batch_shape(self) -> tuple[int, ...]:
        return self.logits.shape[:-1]

    @property
    def probs(self) -> Array:
        return softmax(self.logits, axis=-1)

    @property
    def log_probs(self) -> Array:
        return log_softmax(self.logits, axis=-1)

    def __add__(self, other: "CategoricalNatParams") -> "CategoricalNatParams":
        return CategoricalNatParams(logits=self.logits + other.logits)

    def __sub__(self, other: "CategoricalNatParams") -> "CategoricalNatParams":
        return CategoricalNatParams(logits=self.logits - other.logits)

    def __mul__(self, alpha: float) -> "CategoricalNatParams":
        return CategoricalNatParams(logits=self.logits * alpha)

    def __rmul__(self, alpha: float) -> "CategoricalNatParams":
        return self.__mul__(alpha)

    def __getitem__(self, s) -> "CategoricalNatParams":
        return CategoricalNatParams(logits=self.logits[s])

    def sum(self, batch_axis: int) -> "CategoricalNatParams":
        return CategoricalNatParams(logits=self.logits.sum(batch_axis))


def categorical_cross_entropy(
    p: CategoricalNatParams,
    q: CategoricalNatParams,
) -> Array:
    """Expected log-likelihood ``E_p[log q]`` (the negative cross-entropy).

    Note this returns ``sum_x p(x) log q(x)``, i.e. the *negative* of the
    cross-entropy ``H(p, q)``; ``-categorical_cross_entropy(q, q)`` is the
    Shannon entropy of ``q``.
    """
    return jnp.sum(p.probs * q.log_probs, axis=-1)


print("dists defined.")


## Part 2 — Utilities (`utils`)

Small array helpers and a Flax-style initializer used by the RPM prior.


In [ ]:
def batched_outer_product(A: Array, B: Array) -> Array:
    """Batched outer product ``C[idx, d1, d2] = A[idx, d1] * B[idx, d2]``.

    ``idx`` indexes all leading (batch) dimensions, which must match exactly
    between ``A`` and ``B``; the last axis is the event dimension.
    """
    batch_shape_A = A.shape[:-1]
    batch_shape_B = B.shape[:-1]

    dim_event_a = A.shape[-1]
    dim_event_b = B.shape[-1]

    if batch_shape_A != batch_shape_B:
        raise ValueError(
            f"Batch dimensions must agree. Got full shapes {A.shape}, {B.shape}"
        )

    new_shape = batch_shape_A + (dim_event_a, dim_event_b)

    # Flatten the batch dims, take a per-row outer product, then restore shape.
    a_flat = jnp.reshape(A, (-1, dim_event_a))
    b_flat = jnp.reshape(B, (-1, dim_event_b))
    out_flat = vmap(jnp.outer, in_axes=[0, 0])(a_flat, b_flat)

    return jnp.reshape(out_flat, new_shape)


def batch_identity_init(key: Any, batch: int | Sequence[int], dim: int) -> Array:
    """Initializer: a ``(*batch, dim, dim)`` stack of identity matrices.

    Matches the flax initializer calling convention; ``key`` is ignored.
    """
    if isinstance(batch, int):
        batch = (batch,)
    else:
        batch = tuple(batch)

    shape = batch + (dim, dim)
    M = jnp.zeros(shape)
    return M.at[..., jnp.arange(dim), jnp.arange(dim)].set(1.0)


def identity_init(key: Any, dim: Sequence[int], scale: float = 2**-0.5) -> Array:
    """Initializer: a ``(dim[0], dim[0])`` identity matrix scaled by ``scale``.

    ``dim`` is the flax-style shape tuple (only its first entry is used);
    ``key`` is ignored.
    """
    return scale * jnp.eye(dim[0])


print("utils defined.")


## Part 3 — Recognition networks (`networks`)

MLP / CNN / ResNet / ViT backbones. Each ``__call__`` returns
``(output, aux)`` where ``aux = {"trunk": ..., "projection": ...}`` — the
pre-projection representation (SimCLR's ``h``) and the post-projection one
(SimCLR's ``z``). The Gaussian-output classes feed the projection through a
final natural-parameter head; the Categorical ones feed it through a logits
head instead.

(``LinearRecognition``/``UnnormalizedRecognition`` from the original file are
omitted here: the latter is dead code — unused by any ``encoder_arch``
dispatch and its ``lognormalizer`` deliberately raises
``NotImplementedError`` — and the former has no trunk/projection concept at
all, so neither is reachable by the two experiments below.)


In [ ]:
Aux = Mapping[str, Any]


def vec2tri(v: Array, n: int) -> Array:
    """Scatter a vector into the strictly-lower-triangular part of an ``n x n``
    matrix (the diagonal and upper triangle stay zero).

    ``v`` must have length ``n * (n - 1) / 2``.
    """
    M = jnp.zeros((n, n))
    idxs = jnp.tril_indices(n, -1, n)
    M = M.at[idxs].set(v)
    return M


def gaussian_nat_head(
        x: Array,
        dim_out: int,
        precision_type: str = "full"
) -> DiagGaussianNatParams | GaussianNatParams:
    pwm = nn.Dense(dim_out, name="pwm head")(x)

    if precision_type == "fixed_diag":
        p_diag = jnp.ones(dim_out)
        p_diag = jnp.broadcast_to(p_diag, pwm.shape)
        return DiagGaussianNatParams(precision_diag=p_diag,
                                     precision_weighted_mean=pwm)
    elif precision_type == "diag":
        p_diag = nn.softplus(nn.Dense(dim_out, name="precision diag head")(x))
        return DiagGaussianNatParams(precision_diag=p_diag,
                                     precision_weighted_mean=pwm)
    elif precision_type == "full":
        L_diag = nn.softplus(nn.Dense(dim_out, name="precision diag head")(x))
        L_offd = nn.Dense((dim_out * (dim_out - 1)) // 2,
                          name="precision offdiag head")(x)
        L = vmap(jit(vec2tri, static_argnums=1), in_axes=[0, None])(
            L_offd, dim_out) + vmap(jnp.diag, in_axes=0)(L_diag)
        p = jnp.matmul(L, L.transpose(0, 2, 1))
        return GaussianNatParams(precision=p, precision_weighted_mean=pwm)
    else:
        raise ValueError(f"Invalid 'precision_type': {precision_type}")


class NNRecognition(nn.Module):
    """MLP recognition network (flattens its input)."""

    features: Sequence[int]
    dim_out: int
    projection_features: Sequence[int] = ()
    precision_type: str = "diag"

    @property
    def trunk_dim(self):
        return self.features[-1]

    @property
    def projection_dim(self):
        return self.projection_features[-1]

    @property
    def latent_dim(self):
        return self.dim_out

    @nn.compact
    def __call__(
            self, inputs: Array
    ) -> tuple[GaussianNatParams | DiagGaussianNatParams, Aux]:
        x = inputs
        x = x.reshape((x.shape[0], -1))
        for i, feat in enumerate(self.features):
            x = nn.Dense(feat, name=f"Trunk layer {i}")(x)
            x = nn.relu(x)

        trunk = x

        # Projection layers
        for i, d in enumerate(self.projection_features):
            x = nn.Dense(d, name=f"projection_{i}")(x)
            x = nn.relu(x)

        projection = x

        aux = {"trunk": trunk, "projection": projection}

        nat_params = gaussian_nat_head(x, self.dim_out, self.precision_type)

        return nat_params, aux


class CNNRecognition(nn.Module):
    """Convolutional recognition network (conv stack -> MLP head)."""

    conv_features: Sequence[Tuple[int, Tuple[int, int]]]
    fc_features: Sequence[int]
    dim_out: int
    projection_features: Sequence[int] = ()
    precision_type: str = "diag"

    @property
    def trunk_dim(self):
        return self.fc_features[-1]

    @property
    def projection_dim(self):
        return self.projection_features[-1]

    @property
    def latent_dim(self):
        return self.dim_out

    @nn.compact
    def __call__(
            self, inputs: Array
    ) -> tuple[GaussianNatParams | DiagGaussianNatParams, Aux]:
        x = inputs
        for i, (channels, kernel_size) in enumerate(self.conv_features):
            x = nn.Conv(channels, kernel_size, name=f"conv {i}")(x)
            x = nn.relu(x)
            x = nn.avg_pool(x, window_shape=(2, 2), strides=(2, 2))

        x = x.reshape((x.shape[0], -1))
        for i, d in enumerate(self.fc_features):
            x = nn.Dense(d, name=f"fc {i}")(x)
            x = nn.relu(x)

        trunk = x

        # Projection layers
        for i, d in enumerate(self.projection_features):
            x = nn.Dense(d, name=f"projection_{i}")(x)
            x = nn.relu(x)

        projection = x

        nat_params = gaussian_nat_head(x, self.dim_out, self.precision_type)

        aux = {'trunk': trunk, 'projection': projection}
        return nat_params, aux


class ResidualBlock(nn.Module):
    """Pre-activation-style residual block with GroupNorm."""

    channels: int
    stride: int = 1

    @nn.compact
    def __call__(self, inputs: Array) -> Array:
        x = inputs
        residual = inputs

        x = nn.Conv(
            self.channels,
            (3, 3),
            strides=(self.stride, self.stride),
            padding="SAME",
            use_bias=False,
        )(x)
        x = nn.GroupNorm(num_groups=min(32, self.channels))(x)
        x = nn.relu(x)

        x = nn.Conv(self.channels, (3, 3), padding="SAME", use_bias=False)(x)
        x = nn.GroupNorm(num_groups=min(32, self.channels))(x)

        # Project the skip connection when shape / stride changes.
        if residual.shape[-1] != self.channels or self.stride != 1:
            residual = nn.Conv(
                self.channels,
                (1, 1),
                strides=(self.stride, self.stride),
                use_bias=False,
            )(residual)
            residual = nn.GroupNorm(num_groups=min(32, self.channels))(residual)

        return nn.relu(x + residual)


class BottleneckBlock(nn.Module):
    """Pre-activation ResNet bottleneck block with GroupNorm."""

    channels: int    # base channels (e.g., 64, 128, ...)
    stride: int = 1
    expansion: int = 4

    @nn.compact
    def __call__(self, inputs: Array) -> Array:
        residual = inputs
        x = inputs

        out_channels = self.channels * self.expansion

        # 1x1 reduce
        x = nn.Conv(
            self.channels,
            (1, 1),
            strides=(1, 1),
            use_bias=False,
        )(x)
        x = nn.GroupNorm(num_groups=min(32, self.channels))(x)
        x = nn.relu(x)

        # 3x3 (does downsampling via stride if needed)
        x = nn.Conv(
            self.channels,
            (3, 3),
            strides=(self.stride, self.stride),
            padding="SAME",
            use_bias=False,
        )(x)
        x = nn.GroupNorm(num_groups=min(32, self.channels))(x)
        x = nn.relu(x)

        # 1x1 expand
        x = nn.Conv(
            out_channels,
            (1, 1),
            strides=(1, 1),
            use_bias=False,
        )(x)
        x = nn.GroupNorm(num_groups=min(32, out_channels))(x)

        # Projection if needed
        if residual.shape[-1] != out_channels or self.stride != 1:
            residual = nn.Conv(
                out_channels,
                (1, 1),
                strides=(self.stride, self.stride),
                use_bias=False,
            )(residual)
            residual = nn.GroupNorm(num_groups=min(32, out_channels))(residual)

        return nn.relu(x + residual)


class ResNetRecognition(nn.Module):
    """ResNet-style recognition network (stem -> residual stages -> head)."""

    stage_sizes: Sequence[int]
    stage_widths: Sequence[int]
    dim_out: int
    stem_width: int = 64
    stem_kernel_size: Tuple[int, int] = (7, 7)
    stem_stride: int = 2
    use_max_pool: bool = True
    max_pool_window: Tuple[int, int] = (3, 3)
    max_pool_stride: Tuple[int, int] = (2, 2)
    block_type: str = "basic"
    fc_features: Sequence[int] = ()
    projection_features: Sequence[int] = ()
    precision_type: str = "full"

    @property
    def trunk_dim(self):
        if self.fc_features:
            return self.fc_features[-1]
        # No FC layers after the backbone: the trunk is the raw
        # global-average-pooled output. Bottleneck blocks expand the last
        # stage width by 4x (matches ``BottleneckBlock.expansion``).
        expansion = 4 if self.block_type == "bottleneck" else 1
        return self.stage_widths[-1] * expansion

    @property
    def projection_dim(self):
        return self.projection_features[-1]

    @property
    def latent_dim(self):
        return self.dim_out

    @nn.compact
    def __call__(
            self, inputs: Array
    ) -> tuple[GaussianNatParams | DiagGaussianNatParams, Aux]:
        if len(self.stage_sizes) != len(self.stage_widths):
            raise ValueError(
                "'stage_sizes' and 'stage_widths' must have the same length")
        if self.block_type == "basic":
            Block = ResidualBlock
        elif self.block_type == "bottleneck":
            Block = BottleneckBlock
        else:
            raise ValueError(
                "'block_type' must be 'basic' or 'bottleneck', got "
                f"{self.block_type!r}")

        x = inputs
        x = nn.Conv(
            self.stem_width,
            self.stem_kernel_size,
            strides=(self.stem_stride, self.stem_stride),
            padding="SAME",
            use_bias=False,
            name="stem_conv",
        )(x)
        x = nn.GroupNorm(num_groups=min(32, self.stem_width),
                         name="stem_norm")(x)
        x = nn.relu(x)
        if self.use_max_pool:
            x = nn.max_pool(
                x,
                window_shape=self.max_pool_window,
                strides=self.max_pool_stride,
                padding="SAME",
            )

        for stage_idx, (num_blocks, channels) in enumerate(
                zip(self.stage_sizes, self.stage_widths)):
            for block_idx in range(num_blocks):
                # Downsample at the first block of every stage after the first.
                stride = 2 if stage_idx > 0 and block_idx == 0 else 1
                x = Block(
                    channels=channels,
                    stride=stride,
                    name=f"stage_{stage_idx}_block_{block_idx}",
                )(x)

        # Global average pool over the spatial dims.
        x = jnp.mean(x, axis=(1, 2))
        for i, d in enumerate(self.fc_features):
            x = nn.Dense(d, name=f"fc_{i}")(x)
            x = nn.relu(x)

        trunk = x

        # Projection layers
        for i, d in enumerate(self.projection_features):
            x = nn.Dense(d, name=f"projection_{i}")(x)
            x = nn.relu(x)

        projection = x

        nat_params = gaussian_nat_head(x, self.dim_out, self.precision_type)

        aux = {'trunk': trunk, 'projection': projection}

        return nat_params, aux


class TransformerBlock(nn.Module):
    """Standard pre-norm transformer encoder block."""

    embed_dim: int
    mlp_dim: int
    num_heads: int

    @nn.compact
    def __call__(self, inputs: Array) -> Array:
        x = inputs
        h = nn.LayerNorm()(x)
        h = nn.MultiHeadDotProductAttention(
            num_heads=self.num_heads,
            qkv_features=self.embed_dim,
            out_features=self.embed_dim,
            deterministic=True,
        )(h, h)
        x = x + h

        h = nn.LayerNorm()(x)
        h = nn.Dense(self.mlp_dim)(h)
        h = nn.gelu(h)
        h = nn.Dense(self.embed_dim)(h)

        return x + h


class ViTRecognition(nn.Module):
    """Vision-transformer recognition network."""

    image_size: int
    patch_size: int
    embed_dim: int
    depth: int
    num_heads: int
    mlp_dim: int
    dim_out: int
    representation_dim: int = 0
    projection_features: Sequence[int] = ()
    precision_type: str = "full"

    @property
    def trunk_dim(self):
        # Mirrors __call__: the trunk is the mean-pooled token width
        # (embed_dim), unless a representation head projects it down/up to
        # representation_dim first.
        return (self.representation_dim
                if self.representation_dim > 0 else self.embed_dim)

    @property
    def projection_dim(self):
        return self.projection_features[-1]

    @property
    def latent_dim(self):
        return self.dim_out

    @nn.compact
    def __call__(
            self, inputs: Array
    ) -> tuple[GaussianNatParams | DiagGaussianNatParams, Aux]:
        if self.image_size % self.patch_size != 0:
            raise ValueError("'image_size' must be divisible by 'patch_size'")

        x = inputs
        # Patch embedding via a strided convolution.
        x = nn.Conv(
            self.embed_dim,
            kernel_size=(self.patch_size, self.patch_size),
            strides=(self.patch_size, self.patch_size),
            padding="VALID",
            name="patch_embed",
        )(x)

        batch, h, w, c = x.shape
        x = x.reshape((batch, h * w, c))

        pos_emb = self.param(
            "position_embedding",
            nn.initializers.normal(stddev=0.02),
            (1, h * w, self.embed_dim),
        )
        x = x + pos_emb

        for i in range(self.depth):
            x = TransformerBlock(
                embed_dim=self.embed_dim,
                mlp_dim=self.mlp_dim,
                num_heads=self.num_heads,
                name=f"transformer_block_{i}",
            )(x)

        x = nn.LayerNorm(name="final_norm")(x)
        # Mean-pool over tokens.
        x = jnp.mean(x, axis=1)

        if self.representation_dim > 0:
            x = nn.Dense(self.representation_dim, name="representation_head")(x)
            x = nn.tanh(x)

        trunk = x

        # Projection layers
        for i, d in enumerate(self.projection_features):
            x = nn.Dense(d, name=f"projection_{i}")(x)
            x = nn.relu(x)

        projection = x

        nat_params = gaussian_nat_head(x, self.dim_out, self.precision_type)

        aux = {'trunk': trunk, 'projection': projection}
        return nat_params, aux


# --- Categorical recognition networks ---


class CategoricalCNNRecognition(nn.Module):
    """CNN recognition network that outputs categorical logits."""

    conv_features: Sequence[Tuple[int, Tuple[int, int]]]
    fc_features: Sequence[int]
    dim_out: int

    @nn.compact
    def __call__(self, inputs: Array) -> tuple[CategoricalNatParams, Aux]:
        x = inputs
        for i, (channels, kernel_size) in enumerate(self.conv_features):
            x = nn.Conv(channels, kernel_size, name=f"conv_{i}")(x)
            x = nn.relu(x)
            x = nn.avg_pool(x, window_shape=(2, 2), strides=(2, 2))

        x = x.reshape((x.shape[0], -1))
        for i, d in enumerate(self.fc_features):
            x = nn.Dense(d, name=f"fc_{i}")(x)
            x = nn.relu(x)

        logits = nn.Dense(self.dim_out, name="logits_head")(x)
        logits = nn.log_softmax(logits, axis=-1)
        return CategoricalNatParams(logits=logits), {}


class CategoricalNNRecognition(nn.Module):
    """MLP recognition network that outputs categorical logits."""

    features: Sequence[int]
    dim_out: int

    @nn.compact
    def __call__(self, inputs: Array) -> tuple[CategoricalNatParams, Aux]:
        x = inputs
        # Flatten any spatial / channel dims so image-shaped views are
        # accepted (mirrors NNRecognition).
        x = x.reshape((x.shape[0], -1))
        for i, feat in enumerate(self.features):
            x = nn.Dense(feat, name=f"fc_{i}")(x)
            x = nn.relu(x)

        logits = nn.Dense(self.dim_out, name="logits_head")(x)
        logits = nn.log_softmax(logits, axis=-1)
        return CategoricalNatParams(logits=logits), {}


class CategoricalResNetRecognition(nn.Module):
    """ResNet recognition network that outputs categorical logits."""

    stage_sizes: Sequence[int]
    stage_widths: Sequence[int]
    dim_out: int
    stem_width: int = 64
    stem_kernel_size: Tuple[int, int] = (7, 7)
    stem_stride: int = 2
    use_max_pool: bool = True
    max_pool_window: Tuple[int, int] = (3, 3)
    max_pool_stride: Tuple[int, int] = (2, 2)
    fc_features: Sequence[int] = ()

    @nn.compact
    def __call__(self, inputs: Array) -> tuple[CategoricalNatParams, Aux]:
        if len(self.stage_sizes) != len(self.stage_widths):
            raise ValueError(
                "'stage_sizes' and 'stage_widths' must have the same length")

        x = inputs
        x = nn.Conv(
            self.stem_width,
            self.stem_kernel_size,
            strides=(self.stem_stride, self.stem_stride),
            padding="SAME",
            use_bias=False,
            name="stem_conv",
        )(x)
        x = nn.GroupNorm(num_groups=min(32, self.stem_width),
                         name="stem_norm")(x)
        x = nn.relu(x)
        if self.use_max_pool:
            x = nn.max_pool(
                x,
                window_shape=self.max_pool_window,
                strides=self.max_pool_stride,
                padding="SAME",
            )

        for stage_idx, (num_blocks, channels) in enumerate(
                zip(self.stage_sizes, self.stage_widths)):
            for block_idx in range(num_blocks):
                # Downsample at the first block of every stage after the first.
                stride = 2 if stage_idx > 0 and block_idx == 0 else 1
                x = ResidualBlock(
                    channels=channels,
                    stride=stride,
                    name=f"stage_{stage_idx}_block_{block_idx}",
                )(x)

        # Global average pool over the spatial dims.
        x = jnp.mean(x, axis=(1, 2))
        for i, d in enumerate(self.fc_features):
            x = nn.Dense(d, name=f"fc_{i}")(x)
            x = nn.relu(x)

        logits = nn.Dense(self.dim_out, name="logits_head")(x)
        logits = nn.log_softmax(logits, axis=-1)
        return CategoricalNatParams(logits=logits), {}


class CategoricalViTRecognition(nn.Module):
    """Vision-transformer recognition network that outputs categorical logits."""

    image_size: int
    patch_size: int
    embed_dim: int
    depth: int
    num_heads: int
    mlp_dim: int
    dim_out: int
    representation_dim: int = 0

    @nn.compact
    def __call__(self, inputs: Array) -> tuple[CategoricalNatParams, Aux]:
        if self.image_size % self.patch_size != 0:
            raise ValueError("'image_size' must be divisible by 'patch_size'")

        x = inputs
        # Patch embedding via a strided convolution.
        x = nn.Conv(
            self.embed_dim,
            kernel_size=(self.patch_size, self.patch_size),
            strides=(self.patch_size, self.patch_size),
            padding="VALID",
            name="patch_embed",
        )(x)

        batch, h, w, c = x.shape
        x = x.reshape((batch, h * w, c))

        pos_emb = self.param(
            "position_embedding",
            nn.initializers.normal(stddev=0.02),
            (1, h * w, self.embed_dim),
        )
        x = x + pos_emb

        for i in range(self.depth):
            x = TransformerBlock(
                embed_dim=self.embed_dim,
                mlp_dim=self.mlp_dim,
                num_heads=self.num_heads,
                name=f"transformer_block_{i}",
            )(x)

        x = nn.LayerNorm(name="final_norm")(x)
        # Mean-pool over tokens.
        x = jnp.mean(x, axis=1)

        if self.representation_dim > 0:
            x = nn.Dense(self.representation_dim, name="representation_head")(x)
            x = nn.tanh(x)

        logits = nn.Dense(self.dim_out, name="logits_head")(x)
        logits = nn.log_softmax(logits, axis=-1)
        return CategoricalNatParams(logits=logits), {}


print("networks defined.")


## Part 4 — RPM models (`rpms`)

The Gaussian RPM (two auxiliary constructions: `"constrained"` and
`"amortized"`) and the Categorical RPM, plus their free-energy objectives.
Every model's `__call__` returns `(rpm_output, aux)`: `rpm_output` is a dict
of the distributions the free-energy math needs, and `aux` is whatever the
underlying encoder produced (`{"trunk": ..., "projection": ...}`).


In [ ]:
EncoderParams = Mapping[str, Any]
Params = VariableDict

GaussianLike = GaussianNatParams | DiagGaussianNatParams
RPMOutput = Mapping[str, Any]


def _gaussian_encoder_cls(encoder_arch: str) -> type[nn.Module]:
    """Map an ``encoder_arch`` string to its recognition-network class."""
    if encoder_arch == "nn":
        return NNRecognition
    elif encoder_arch == "cnn":
        return CNNRecognition
    elif encoder_arch == "resnet":
        return ResNetRecognition
    elif encoder_arch == "vit":
        return ViTRecognition
    else:
        raise ValueError(
            "Invalid 'encoder_arch'; must be 'nn', 'cnn', "
            f"'resnet', or 'vit', got {encoder_arch}")


class GaussianRPM(nn.Module):
    """Abstract Gaussian RPM. Use :meth:`create` to build a concrete model."""

    n_factors: int
    dim_latent: int
    encoder_arch: str
    encoder_params: EncoderParams | None = None
    share_recognition: bool = False
    precision_type: str = "full"
    fix_prior: bool = False
    condition_on_view_info: bool = False

    @property
    def rpm_type(self) -> str:
        return "gaussian"

    @property
    def _unbound_encoder(self) -> nn.Module:
        """A plain (non-vmapped) encoder instance, usable without binding.

        ``self.encoders`` is only assigned in ``setup()``, so it only exists
        once this module is bound (inside ``init``/``apply``). ``trunk_dim``
        and ``latent_dim`` only need the encoder's static dataclass fields, so
        this sidesteps binding entirely by constructing an unwrapped instance.
        """
        encoder_cls = _gaussian_encoder_cls(self.encoder_arch)
        return encoder_cls(
            **self.encoder_params,
            dim_out=self.dim_latent,
            precision_type=self.precision_type,
        )

    @property
    def trunk_dim(self):
        return self._unbound_encoder.trunk_dim

    @property
    def latent_dim(self):
        return self._unbound_encoder.latent_dim

    def __init__(self, *args):
        raise NotImplementedError(
            "Cannot instantiate the abstract class, use GaussianRPM.create instead"
        )

    def __call__(self, inputs: Array) -> tuple[RPMOutput, Aux]:
        raise NotImplementedError(
            "Cannot apply abstract class, use GaussianRPM.create to get a concrete model"
        )

    def setup(self):
        Encoder = _gaussian_encoder_cls(self.encoder_arch)

        # One recognition network per factor, vmapped over the factor axis
        # (input axis 1). Params are shared across factors iff ``share_recognition``.
        VmapEncoders = nn.vmap(
            Encoder,
            in_axes=1,
            out_axes=0,
            variable_axes=({
                "params": None if self.share_recognition else 0,
                "intermediates": 0
            }),
            split_rngs=({
                "params": not self.share_recognition
            }),
        )
        self.encoders = VmapEncoders(
            **self.encoder_params,
            dim_out=self.dim_latent,
            precision_type=self.precision_type,
        )

        # Prior natural params. When ``fix_prior`` the prior is a constant
        # standard Gaussian; otherwise it is learned.
        if self.fix_prior:
            self.prior_pwm = jnp.zeros((self.dim_latent,))
            if self.precision_type == "diag":
                self.prior_log_precision_diag = jnp.zeros((self.dim_latent,))
            else:
                self.prior_cholesky = identity_init(None, [self.dim_latent])
        else:
            self.prior_pwm = self.param("prior_pwm", nn.initializers.zeros,
                                        (self.dim_latent,))
            if self.precision_type == "diag":
                self.prior_log_precision_diag = self.param(
                    "prior_log_precision_diag",
                    nn.initializers.zeros,
                    (self.dim_latent,),
                )
            else:
                self.prior_cholesky = self.param("prior_cholesky",
                                                 identity_init,
                                                 (self.dim_latent,))

    def prior(self) -> GaussianLike:
        if self.precision_type == "diag":
            return DiagGaussianNatParams(
                precision_diag=jnp.exp(self.prior_log_precision_diag),
                precision_weighted_mean=self.prior_pwm,
            )
        return GaussianNatParams(
            precision=jnp.dot(self.prior_cholesky, self.prior_cholesky.T),
            precision_weighted_mean=self.prior_pwm,
        )

    @classmethod
    def create(
        cls,
        auxiliary_method: str,
        n_factors: int,
        dim_latent: int,
        encoder_arch: str,
        encoder_params: EncoderParams = {},
        share_recognition: bool = False,
        precision_type: str = "full",
        fix_prior: bool = False,
        n_samples: int = 1,
    ) -> "GaussianRPM":
        """Build a concrete Gaussian RPM for the chosen auxiliary method."""
        if auxiliary_method not in [
                "constrained", "amortized", "reparametrised"
        ]:
            raise ValueError(
                "'auxiliary_method' has to be either 'constrained', "
                "'optimized', 'amortized', or 'reparametrised'")

        kwargs = dict(
            n_factors=n_factors,
            dim_latent=dim_latent,
            encoder_arch=encoder_arch,
            encoder_params=encoder_params,
            share_recognition=share_recognition,
            precision_type=precision_type,
            fix_prior=fix_prior,
        )

        if auxiliary_method == "constrained":
            return GaussianRPM_ConstrainedAux(**kwargs)
        elif auxiliary_method == "amortized":
            return GaussianRPM_AmortizedAux(**kwargs)
        else:
            raise NotImplementedError(
                f"Unknown auxiliary_method={auxiliary_method!r}")


class GaussianRPM_AmortizedAux(GaussianRPM):
    """RPM whose auxiliary is produced by a second set of recognition nets."""

    def setup(self):
        super().setup()

        AuxEncoder = _gaussian_encoder_cls(self.encoder_arch)

        VmapAuxEncoders = nn.vmap(
            AuxEncoder,
            in_axes=1,
            out_axes=0,
            variable_axes={
                "params": 0,
                "intermediates": 0
            },
            split_rngs={"params": True},
        )
        self.aux_encoders = VmapAuxEncoders(
            **self.encoder_params,
            dim_out=self.dim_latent,
            precision_type=self.precision_type,
        )

    def __call__(self, inputs: Array) -> tuple[RPMOutput, Aux]:
        prior = self.prior()
        recognition, aux = self.encoders(inputs)
        auxiliary, _ = self.aux_encoders(inputs)
        variational = (1 / (self.n_factors + 1)) * (
            prior + auxiliary.sum(batch_axis=0) + recognition.sum(batch_axis=0))

        if self.precision_type == "diag":
            variational_mean = diag_mean_params(variational)
        else:
            variational_mean = mean_params(variational)

        rpm_output = {
            "prior": prior,
            "recognition": recognition,
            "auxiliary": auxiliary,
            "variational": variational,
            "variational_mean": variational_mean,
        }

        return rpm_output, aux


class GaussianRPM_ConstrainedAux(GaussianRPM):
    """RPM whose auxiliary is the closed-form ``variational - prior``."""

    def __call__(self, inputs: Array) -> tuple[RPMOutput, Aux]:
        x = inputs
        J = x.shape[1]
        if J != self.n_factors:
            raise ValueError(
                "Second input dimension must agree with number of factors; "
                f"RPM has n_factors={self.n_factors}, input has shape {x.shape}"
            )

        prior = self.prior()
        deltas, aux = self.encoders(x)
        variational = prior + deltas.sum(batch_axis=0)
        recognition = prior + deltas
        auxiliary = variational - prior

        if self.precision_type == "diag":
            variational_mean = diag_mean_params(variational)
        else:
            variational_mean = mean_params(variational)

        rpm_output = {
            "prior": prior,
            "recognition": recognition,
            "auxiliary": auxiliary,
            "variational": variational,
            "variational_mean": variational_mean,
        }

        return rpm_output, aux


def free_energy(
    params: Params,
    model: GaussianRPM,
    X: Array,
    rng: Array | None = None,
    beta: float = 1.0,
) -> tuple[Array, Aux]:
    """RPM auxiliary free energy"""
    # Prior, per-factor recognition, auxiliary, and the inferred posterior.
    del rng    # Unused: neither Gaussian RPM variant samples internally.
    rpm_output, _ = model.apply(params, X)

    prior = rpm_output["prior"]
    recognition = rpm_output["recognition"]
    auxiliary = rpm_output["auxiliary"]
    variational = rpm_output["variational"]
    variational_mean = rpm_output["variational_mean"]

    N = variational.batch_shape()[0]
    J = recognition.batch_shape()[0]

    is_diag = isinstance(variational, DiagGaussianNatParams)
    kl_fn = diag_kl if is_diag else kl
    unkl_fn = diag_unnormalized_kl if is_diag else unnormalized_kl
    variational_cov = variational_mean.var_diag if is_diag else variational_mean.cov
    natparam_cls = DiagGaussianNatParams if is_diag else GaussianNatParams

    if len(auxiliary.batch_shape()) == 1:
        auxiliary = auxiliary.expand_batch_dim(0)

    outsum_gaussians = vmap(
        vmap(natparam_cls.__add__, in_axes=[1, None]),
        in_axes=[None, 1],
    )
    outsum_arrays = vmap(vmap(jnp.add, in_axes=[1, None]), in_axes=[None, 1])

    # has now batch dimension of [M x N x J]
    cross_terms = outsum_gaussians(auxiliary, recognition)

    phi_recognition = recognition.lognormalizer()    # [J x M]
    phi_auxiliary = auxiliary.lognormalizer()    # [1 x N] or [J x N]
    phi_cross_terms = cross_terms.lognormalizer()    # [M x N x J]

    # [M x N x J] -> [N x J] -> []
    gamma_terms = phi_cross_terms - outsum_arrays(phi_auxiliary,
                                                  phi_recognition)
    log_gamma_jn = logsumexp(gamma_terms, axis=1, b=(1 / N))
    logGamma = log_gamma_jn.sum()

    phi_variational = variational.lognormalizer(mean=variational_mean.mean)

    # -KL(q || prior) and the summed -unnormalized-KL factor terms, with the
    # full vs diagonal Gaussian implementations.
    neg_kl_qp = -kl_fn(variational_mean.mean, variational_cov, variational,
                       prior, phi_variational).sum()

    def sum_kl_factors(j, carry):
        ukl = unkl_fn(
            variational_mean.mean,
            variational_cov,
            variational,
            recognition[j],
            auxiliary,
            phi_recognition[j],
            phi_auxiliary,
            phi_variational,
        ).sum()
        return carry - ukl

    q_z_var_mean = jnp.mean(variational_cov)
    q_z_mean_std = jnp.mean(jnp.std(variational_mean.mean, axis=0))

    neg_kl_qf = fori_loop(0, J, sum_kl_factors, 0.0)

    free_energy = (neg_kl_qf + beta * neg_kl_qp - logGamma) / N
    metrics = {
        "neg_kl_qp": neg_kl_qp / N,
        "neg_kl_qf": neg_kl_qf / N,
        "neg_log_gamma": -logGamma / N,
        "q_z_mean_std": q_z_mean_std,
        "q_z_var_mean": q_z_var_mean,
    }

    return free_energy, metrics


# --- Categorical RPM ---


class CategoricalRPM(nn.Module):
    """Categorical RPM with a learned prior over ``num_categories`` classes."""

    n_factors: int
    num_categories: int
    encoder_arch: str
    encoder_params: EncoderParams | None = None
    share_recognition: bool = True
    fix_prior: bool = False

    @property
    def rpm_type(self) -> str:
        return "categorical"

    def setup(self):
        params = self.encoder_params or {}

        if self.encoder_arch == "cnn":
            Encoder = CategoricalCNNRecognition
        elif self.encoder_arch == "nn":
            Encoder = CategoricalNNRecognition
        elif self.encoder_arch == "resnet":
            Encoder = CategoricalResNetRecognition
        elif self.encoder_arch == "vit":
            Encoder = CategoricalViTRecognition
        else:
            raise ValueError(f"Unknown encoder_arch={self.encoder_arch}")

        VmapEncoder = nn.vmap(
            Encoder,
            in_axes=1,
            out_axes=0,
            variable_axes=({
                "params": 0
            } if not self.share_recognition else {
                "params": None
            }),
            split_rngs=({
                "params": True
            } if not self.share_recognition else {
                "params": False
            }),
        )

        self.encoders = VmapEncoder(**params, dim_out=self.num_categories)

        # Prior logits over the categories. When ``fix_prior`` the prior is a
        # constant uniform distribution (zeros, not a parameter); otherwise it
        # is learned (also initialized to uniform).
        if self.fix_prior:
            self.prior_logits = jnp.zeros((self.num_categories,))
        else:
            self.prior_logits = self.param(
                "prior_logits",
                nn.initializers.zeros,
                (self.num_categories,),
            )

    def prior(self) -> CategoricalNatParams:
        return CategoricalNatParams(logits=self.prior_logits)

    def __call__(self, inputs: Array) -> tuple[RPMOutput, Aux]:
        x = inputs
        j = x.shape[1]
        if j != self.n_factors:
            raise ValueError(f"Expected {self.n_factors} views, got {j}")
        prior = self.prior()

        factors, aux = self.encoders(x)    # logits: [J x N x num_categories]

        # log f_bar_j(z) = logsumexp_m log f_j(z|x_{j,m}) - log N.
        n = x.shape[0]
        log_denominators = logsumexp(factors.logits, axis=1) - jnp.log(n)

        factor_sum = factors.logits.sum(axis=0)
        denom_sum = log_denominators.sum(axis=0)

        posterior_logits = prior.logits + factor_sum - denom_sum
        variational = CategoricalNatParams(logits=posterior_logits)

        rpm_output = {
            "prior": prior,
            "factors": factors,
            "variational": variational,
        }

        return rpm_output, aux


def free_energy_categorical(
    params: Params,
    model: CategoricalRPM,
    x: Array,
    rng: Array | None = None,
    beta: float = 1.0,
) -> tuple[Array, Aux]:
    """Free energy for the Categorical RPM.

    Returns the per-datapoint free energy plus diagnostics
    ``(beta*entropy, beta*prior cross-term, factor cross-terms, denominator
    cross-terms)`` (each averaged over datapoints).
    """
    del rng    # Unused: CategoricalRPM does not sample internally.
    rpm_output, _ = model.apply(params, x)
    prior = rpm_output["prior"]
    factors = rpm_output["factors"]
    variational = rpm_output["variational"]

    n = x.shape[0]
    j = model.n_factors

    # entropy = H(q) = -E_q[log q].
    entropy = -categorical_cross_entropy(variational, variational)
    prior_xent = categorical_cross_entropy(variational, prior)

    factors_xent = jnp.zeros(n)
    for fj in range(j):
        factor_j = CategoricalNatParams(logits=factors.logits[fj])
        factors_xent = factors_xent + categorical_cross_entropy(
            variational, factor_j)

    # log f_bar_j(z) = logsumexp_m log f_j(z|x_{j,m}) - log N, i.e. the log of
    # the data-averaged recognition factor (the RPM normalizer).
    log_denominators = logsumexp(factors.logits, axis=1) - jnp.log(n)
    denom_xent = jnp.zeros(n)
    for fj in range(j):
        denom_j = CategoricalNatParams(
            logits=jnp.broadcast_to(log_denominators[fj], (
                n, model.num_categories)))
        denom_xent = denom_xent + categorical_cross_entropy(
            variational, denom_j)

    fe_per_sample = beta * (entropy + prior_xent) + factors_xent - denom_xent
    fe = jnp.sum(fe_per_sample) / n

    metrics = {
        "entropy": jnp.sum(beta * entropy) / n,
        "prior_xent": jnp.sum(beta * prior_xent) / n,
        "factors_xent": jnp.sum(factors_xent) / n,
        "denom_xent": jnp.sum(denom_xent) / n,
    }

    return fe, metrics


print("rpms defined.")


## Part 5 — SimCLR encoder and NT-Xent loss (`simclr`)

Reuses the same recognition-network backbones as the RPM models. Unlike RPM,
SimCLR always shares weights across views, so there is no per-factor
``nn.vmap``: views are simply reshaped into the batch dimension, run through
one ordinary forward pass, and reshaped back. The recognition-network
classes above always attach a (Gaussian) natural-parameter head; since
SimCLR never uses it, ``dim_out=1`` with ``precision_type="fixed_diag"``
makes that head a single throwaway ``Dense(1)`` layer — negligible next to
any real backbone, and avoids duplicating the backbone code.


In [ ]:
_DISCARDED_HEAD_DIM_OUT = 1
_DISCARDED_HEAD_PRECISION_TYPE = "fixed_diag"


def _simclr_encoder_cls(encoder_arch: str) -> type[nn.Module]:
    """Map an ``encoder_arch`` string to its recognition-network class.

    Excludes ``"linear"``: there is no ``LinearRecognition`` in this notebook
    (it has no trunk/projection concept), mirroring ``CategoricalRPM``'s
    encoder_arch support.
    """
    if encoder_arch == "nn":
        return NNRecognition
    elif encoder_arch == "cnn":
        return CNNRecognition
    elif encoder_arch == "resnet":
        return ResNetRecognition
    elif encoder_arch == "vit":
        return ViTRecognition
    else:
        raise ValueError("Invalid 'encoder_arch'; must be 'nn', 'cnn', "
                         f"'resnet', or 'vit', got {encoder_arch}")


class SimCLREncoder(nn.Module):
    """Shared-weight multi-view encoder for SimCLR-style contrastive training."""

    n_views: int
    encoder_arch: str
    encoder_params: EncoderParams | None = None
    temperature: float = 0.5

    @property
    def rpm_type(self) -> str:
        return "simclr"

    @property
    def _unbound_encoder(self) -> nn.Module:
        encoder_cls = _simclr_encoder_cls(self.encoder_arch)
        return encoder_cls(
            **self.encoder_params,
            dim_out=_DISCARDED_HEAD_DIM_OUT,
            precision_type=_DISCARDED_HEAD_PRECISION_TYPE,
        )

    @property
    def trunk_dim(self) -> int:
        return self._unbound_encoder.trunk_dim

    @property
    def latent_dim(self) -> int:
        """Width of the projection output (SimCLR's ``z``).

        Falls back to ``trunk_dim`` when ``projection_features`` is empty,
        in which case the projection head is a no-op and z == h.
        """
        encoder = self._unbound_encoder
        if encoder.projection_features:
            return encoder.projection_dim
        return encoder.trunk_dim

    def setup(self):
        Encoder = _simclr_encoder_cls(self.encoder_arch)
        self.encoder = Encoder(
            **self.encoder_params,
            dim_out=_DISCARDED_HEAD_DIM_OUT,
            precision_type=_DISCARDED_HEAD_PRECISION_TYPE,
        )

    def __call__(self, inputs: Array) -> tuple[Array, Aux]:
        x = inputs    # (batch, n_views, H, W, C)
        n_views = x.shape[1]
        if n_views != self.n_views:
            raise ValueError(f"Expected {self.n_views} views, got {n_views}")
        batch = x.shape[0]
        per_example_shape = x.shape[2:]

        # (batch, n_views, ...) -> (n_views, batch, ...) -> (n_views*batch, ...)
        x_flat = jnp.transpose(x, (1, 0) + tuple(range(2, x.ndim)))
        x_flat = x_flat.reshape((n_views * batch,) + per_example_shape)

        _, aux = self.encoder(x_flat)

        def _unflatten(feats: Array) -> Array:
            return feats.reshape((n_views, batch) + feats.shape[1:])

        trunk = _unflatten(aux["trunk"])
        projection = _unflatten(aux["projection"])

        return projection, {"trunk": trunk, "projection": projection}


def nt_xent_loss(
    params,
    model: SimCLREncoder,
    batch_views: Array,
    rng: Array | None = None,
    beta: float = 1.0,
) -> tuple[Array, Aux]:
    """NT-Xent contrastive loss, generalized to n_views >= 2.

    Every other view of the same image is a positive; every embedding from a
    different image is a negative. Reduces exactly to the standard two-view
    NT-Xent loss when ``n_views == 2`` (verified against an independent
    hand-rolled reference implementation).

    Returns ``-nt_xent`` (not ``nt_xent`` directly) to match the sign
    convention shared with ``free_energy``/``free_energy_categorical``:
    ``create_train_step`` computes ``loss = -free_energy``, so returning the
    negated loss here means the actual NT-Xent loss is what gets minimized.
    """
    del rng, beta    # Unused: temperature (not beta) scales the loss, and
    # the encoder has no dropout/sampling.
    projection, _ = model.apply(params, batch_views)    # (V, B, D)

    n_views, batch = projection.shape[0], projection.shape[1]
    n = n_views * batch
    z = projection.reshape((n, projection.shape[-1]))

    z_norm = z / (jnp.linalg.norm(z, axis=-1, keepdims=True) + 1e-8)
    raw_sim = z_norm @ z_norm.T    # (N, N), in [-1, 1]
    scaled_sim = raw_sim / model.temperature

    row_idx = jnp.arange(n)
    is_self = row_idx[:, None] == row_idx[None, :]
    # Row i = view v, example b (since z was flattened as (n_views, batch)).
    # Row i' is a positive for row i iff same example, different view.
    same_example = (row_idx[:, None] % batch) == (row_idx[None, :] % batch)
    positive_mask = same_example & ~is_self
    negative_mask = ~same_example

    masked_sim = jnp.where(is_self, -jnp.inf, scaled_sim)
    log_prob = nn.log_softmax(masked_sim, axis=-1)    # (N, N)

    num_positives = jnp.sum(positive_mask, axis=-1)    # (N,), == n_views - 1
    per_anchor_loss = -jnp.sum(log_prob * positive_mask,
                               axis=-1) / num_positives
    nt_xent = jnp.mean(per_anchor_loss)

    mean_pos_sim = jnp.sum(raw_sim * positive_mask) / jnp.sum(positive_mask)
    mean_neg_sim = jnp.sum(raw_sim * negative_mask) / jnp.sum(negative_mask)
    predicted = jnp.argmax(masked_sim, axis=-1)
    contrastive_acc = jnp.mean(
        jnp.take_along_axis(positive_mask, predicted[:, None], axis=-1))

    metrics = {
        "mean_pos_sim": mean_pos_sim,
        "mean_neg_sim": mean_neg_sim,
        "contrastive_acc": contrastive_acc,
    }

    return -nt_xent, metrics


print("simclr defined.")


## Part 6 — CIFAR-10 data loading

Uses `tensorflow_datasets` to load CIFAR-10 with two view-generation
strategies selectable via `preprocess_type`:

- **`"simclr"`** — SimCLR-style augmentations (random crop, flip, color
  jitter, grayscale, Gaussian blur).
- **`"lejepa_masking"`** — I-JEPA-style multi-block masking (Assran et al.,
  2023).  Instead of hand-crafted augmentations, each view is the normalized
  image with a different set of large rectangular blocks masked out (filled
  with zeros, i.e. the dataset mean post-normalization).  Target blocks are
  sampled at large scale to encourage semantic representations; the context
  (visible) portion is spatially distributed.

Both strategies return `(raw_image, views, label)` as NumPy arrays with
`views` in `[batch, num_views, H, W, 3]` layout.


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

# Prevent TF from grabbing GPU memory (we only use it for data loading).
tf.config.set_visible_devices([], "GPU")

DataBatch = tuple[np.ndarray, np.ndarray, np.ndarray]

CIFAR10_MEAN = np.asarray([0.4914, 0.4822, 0.4465], dtype=np.float32)
CIFAR10_STD = np.asarray([0.2470, 0.2435, 0.2616], dtype=np.float32)


# ---------------------------------------------------------------------------
# Helper: append a visibility-mask channel
# ---------------------------------------------------------------------------

def _append_ones_mask(image: tf.Tensor) -> tf.Tensor:
    """Append an all-ones 4th channel (fully visible) to a [H,W,3] image."""
    h = tf.shape(image)[0]
    w = tf.shape(image)[1]
    ones = tf.ones([h, w, 1], dtype=image.dtype)
    return tf.concat([image, ones], axis=-1)    # [H, W, 4]


# ---------------------------------------------------------------------------
# SimCLR-style augmentations in pure TF ops
# ---------------------------------------------------------------------------

def _random_resized_crop(image: tf.Tensor, size: int = 32) -> tf.Tensor:
    """Random crop with area in [0.08, 1.0] and aspect ratio in [3/4, 4/3],
    then resize to ``size x size``.  Uses TF's built-in
    ``sample_distorted_bounding_box`` which matches torchvision's
    ``RandomResizedCrop`` semantics (10 attempts, centre-crop fallback)."""
    shape = tf.shape(image)
    bbox = tf.constant([0.0, 0.0, 1.0, 1.0], dtype=tf.float32,
                       shape=[1, 1, 4])
    bbox_begin, bbox_size, _ = tf.image.sample_distorted_bounding_box(
        shape,
        bounding_boxes=bbox,
        min_object_covered=0.0,
        aspect_ratio_range=(3.0 / 4.0, 4.0 / 3.0),
        area_range=(0.08, 1.0),
        max_attempts=10,
        use_image_if_no_bounding_boxes=True,
    )
    image = tf.slice(image, bbox_begin, bbox_size)
    image = tf.image.resize(image, [size, size])
    return image


def _color_jitter(image: tf.Tensor,
                  strength: float = 1.0) -> tf.Tensor:
    """Random brightness / contrast / saturation / hue jitter."""
    image = tf.image.random_brightness(image, max_delta=0.8 * strength)
    image = tf.image.random_contrast(image, lower=max(0, 1 - 0.8 * strength),
                                     upper=1 + 0.8 * strength)
    image = tf.image.random_saturation(image,
                                       lower=max(0, 1 - 0.8 * strength),
                                       upper=1 + 0.8 * strength)
    image = tf.image.random_hue(image, max_delta=0.2 * strength)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image


def _random_grayscale(image: tf.Tensor, p: float = 0.2) -> tf.Tensor:
    """Convert to grayscale with probability ``p``."""
    do_gray = tf.random.uniform([]) < p
    gray = tf.image.rgb_to_grayscale(image)          # [H, W, 1]
    gray = tf.tile(gray, [1, 1, 3])                  # [H, W, 3]
    return tf.where(do_gray, gray, image)


def _gaussian_blur(image: tf.Tensor, kernel_size: int = 3,
                   sigma_lo: float = 0.1,
                   sigma_hi: float = 2.0) -> tf.Tensor:
    """Gaussian blur with a random sigma."""
    sigma = tf.random.uniform([], sigma_lo, sigma_hi)
    radius = kernel_size // 2
    x = tf.cast(tf.range(-radius, radius + 1), tf.float32)
    kernel_1d = tf.exp(-0.5 * (x / sigma) ** 2)
    kernel_1d = kernel_1d / tf.reduce_sum(kernel_1d)
    kernel_2d = tf.tensordot(kernel_1d, kernel_1d, axes=0)  # [k, k]
    kernel_2d = kernel_2d[:, :, tf.newaxis, tf.newaxis]      # [k, k, 1, 1]
    kernel_2d = tf.tile(kernel_2d, [1, 1, 3, 1])             # [k, k, 3, 1]

    image_4d = image[tf.newaxis]                             # [1, H, W, 3]
    blurred = tf.nn.depthwise_conv2d(image_4d, kernel_2d,
                                     strides=[1, 1, 1, 1],
                                     padding="SAME")
    return blurred[0]


def simclr_augment_single(image: tf.Tensor,
                          image_size: int = 32,
                          color_jitter_strength: float = 1.0,
                          gaussian_blur: bool = True,
                          pass_mask: bool = False) -> tf.Tensor:
    """Apply one SimCLR augmentation to a float32 [H,W,3] image in [0,1].

    If ``pass_mask`` is True, an all-ones 4th channel is appended
    (indicating all pixels are visible), giving output shape [H,W,4].
    """
    image = _random_resized_crop(image, size=image_size)
    image = tf.image.random_flip_left_right(image)

    # Color jitter with p=0.8
    do_jitter = tf.random.uniform([]) < 0.8
    image = tf.cond(do_jitter,
                    lambda: _color_jitter(image, strength=color_jitter_strength),
                    lambda: image)

    image = _random_grayscale(image, p=0.2)

    if gaussian_blur:
        do_blur = tf.random.uniform([]) < 0.5
        image = tf.cond(do_blur, lambda: _gaussian_blur(image), lambda: image)

    # Normalize
    image = (image - CIFAR10_MEAN) / CIFAR10_STD

    if pass_mask:
        image = _append_ones_mask(image)

    return image


def eval_preprocess(image: tf.Tensor, image_size: int = 32,
                    pass_mask: bool = False) -> tf.Tensor:
    """Deterministic eval preprocessing: resize (if needed) + normalize.

    If ``pass_mask`` is True, an all-ones 4th channel is appended.
    """
    if image_size != 32:
        image = tf.image.resize(image, [image_size, image_size])
    image = (image - CIFAR10_MEAN) / CIFAR10_STD
    if pass_mask:
        image = _append_ones_mask(image)
    return image


# ---------------------------------------------------------------------------
# I-JEPA / LeJEPA-style multi-block masking (Assran et al., 2023)
# ---------------------------------------------------------------------------

def _sample_block_mask(
    h: int,
    w: int,
    scale_range: tuple[float, float] = (0.15, 0.2),
    aspect_ratio_range: tuple[float, float] = (0.75, 1.5),
) -> tf.Tensor:
    """Sample a single rectangular block mask.

    Returns a boolean [H, W] tensor that is True for the masked region.
    The block's area is ``scale * H * W`` and its aspect ratio is sampled
    log-uniformly from ``aspect_ratio_range``.
    """
    area = tf.cast(h * w, tf.float32)
    target_area = tf.random.uniform([], scale_range[0], scale_range[1]) * area

    log_ratio_lo = tf.math.log(aspect_ratio_range[0])
    log_ratio_hi = tf.math.log(aspect_ratio_range[1])
    aspect_ratio = tf.exp(tf.random.uniform([], log_ratio_lo, log_ratio_hi))

    block_h = tf.cast(
        tf.minimum(tf.round(tf.sqrt(target_area / aspect_ratio)),
                   tf.cast(h, tf.float32)),
        tf.int32)
    block_w = tf.cast(
        tf.minimum(tf.round(tf.sqrt(target_area * aspect_ratio)),
                   tf.cast(w, tf.float32)),
        tf.int32)
    block_h = tf.maximum(block_h, 1)
    block_w = tf.maximum(block_w, 1)

    top = tf.random.uniform([], 0, h - block_h + 1, dtype=tf.int32)
    left = tf.random.uniform([], 0, w - block_w + 1, dtype=tf.int32)

    rows = tf.range(h)
    cols = tf.range(w)
    row_mask = (rows >= top) & (rows < top + block_h)     # [H]
    col_mask = (cols >= left) & (cols < left + block_w)    # [W]
    return row_mask[:, tf.newaxis] & col_mask[tf.newaxis, :]  # [H, W]


def _sample_multi_block_mask(
    h: int,
    w: int,
    num_blocks: int = 4,
    scale_range: tuple[float, float] = (0.15, 0.2),
    aspect_ratio_range: tuple[float, float] = (0.75, 1.5),
) -> tf.Tensor:
    """Sample the union of ``num_blocks`` rectangular block masks.

    Returns a boolean [H, W] tensor that is True for *masked* pixels.
    Following I-JEPA, target blocks are sampled at sufficiently large scale
    so that the prediction task is semantic rather than local interpolation.
    """
    mask = tf.zeros([h, w], dtype=tf.bool)
    for _ in range(num_blocks):
        block = _sample_block_mask(h, w, scale_range, aspect_ratio_range)
        mask = mask | block
    return mask


def lejepa_masking_augment_single(
    image: tf.Tensor,
    image_size: int = 32,
    num_blocks: int = 4,
    scale_range: tuple[float, float] = (0.15, 0.2),
    aspect_ratio_range: tuple[float, float] = (0.75, 1.5),
    pass_mask: bool = False,
) -> tf.Tensor:
    """Create one masked view of a float32 [H,W,3] image in [0,1].

    The image is first normalized, then ``num_blocks`` random rectangular
    regions are zeroed out (zero = dataset mean post-normalization).

    If ``pass_mask`` is True, a visibility channel (1=visible, 0=masked) is
    concatenated as the 4th channel, giving output shape [H,W,4].
    """
    if image_size != 32:
        image = tf.image.resize(image, [image_size, image_size])
    image = (image - CIFAR10_MEAN) / CIFAR10_STD

    mask = _sample_multi_block_mask(
        image_size, image_size,
        num_blocks=num_blocks,
        scale_range=scale_range,
        aspect_ratio_range=aspect_ratio_range,
    )  # [H, W], True = masked
    mask_3d = tf.cast(mask[:, :, tf.newaxis], tf.float32)  # [H, W, 1]
    # Replace masked pixels with 0 (= dataset mean after normalization).
    image = image * (1.0 - mask_3d)

    if pass_mask:
        visibility = 1.0 - mask_3d   # [H, W, 1], 1=visible, 0=masked
        image = tf.concat([image, visibility], axis=-1)  # [H, W, 4]

    return image


# ---------------------------------------------------------------------------
# tf.data pipeline
# ---------------------------------------------------------------------------

class TFDSLoader:
    """CIFAR-10 data loader using tensorflow_datasets.

    Yields ``(raw_images, views, labels)`` as NumPy arrays with
    ``views`` in ``[batch, num_views, H, W, C]`` layout (JAX convention),
    where C=3 normally or C=4 when ``pass_mask=True``.
    """

    def __init__(
        self,
        *,
        split: str,
        batch_size: int,
        training: bool,
        num_views: int = 2,
        image_size: int = 32,
        preprocess_type: str = "simclr",
        color_jitter_strength: float = 1.0,
        gaussian_blur: bool = True,
        masking_num_blocks: int = 4,
        masking_scale_range: tuple[float, float] = (0.15, 0.2),
        masking_aspect_ratio_range: tuple[float, float] = (0.75, 1.5),
        pass_mask: bool = False,
        seed: Optional[int] = None,
        data_dir: Optional[str] = None,
        drop_remainder: Optional[bool] = None,
        cache: bool = True,
    ) -> None:
        self._batch_size = batch_size
        self._training = training
        self._num_views = num_views
        self._image_size = image_size
        self._preprocess_type = preprocess_type
        self._color_jitter_strength = color_jitter_strength
        self._gaussian_blur = gaussian_blur
        self._masking_num_blocks = masking_num_blocks
        self._masking_scale_range = masking_scale_range
        self._masking_aspect_ratio_range = masking_aspect_ratio_range
        self._pass_mask = pass_mask
        self._seed = seed
        self._num_classes = 10

        if drop_remainder is None:
            drop_remainder = training
        self._drop_remainder = drop_remainder

        tfds_split = "train" if split == "train" else "test"
        ds_info = tfds.builder("cifar10", data_dir=data_dir).info
        self._dataset_size = ds_info.splits[tfds_split].num_examples

        ds = tfds.load(
            "cifar10",
            split=tfds_split,
            as_supervised=True,       # yields (image, label)
            data_dir=data_dir,
            shuffle_files=training,
        )

        if cache:
            ds = ds.cache()

        if training:
            ds = ds.shuffle(
                buffer_size=min(self._dataset_size, 50_000),
                seed=seed,
                reshuffle_each_iteration=True,
            )

        ds = ds.map(self._preprocess, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=drop_remainder)
        ds = ds.prefetch(tf.data.AUTOTUNE)
        self._ds = ds

    # -- Properties expected by the training loop --

    @property
    def dataset_size(self) -> int:
        return self._dataset_size

    @property
    def num_classes(self) -> int:
        return self._num_classes

    # -- Preprocessing --

    def _preprocess(self, image: tf.Tensor,
                    label: tf.Tensor):
        """Map a single (image, label) example to (raw, views, label)."""
        # raw_image: uint8 [H, W, 3] — unchanged original for visualization.
        raw_image = image

        # Float image in [0, 1] for augmentation.
        image_f = tf.cast(image, tf.float32) / 255.0

        if self._training:
            if self._preprocess_type == "simclr":
                views = tf.stack([
                    simclr_augment_single(
                        image_f,
                        image_size=self._image_size,
                        color_jitter_strength=self._color_jitter_strength,
                        gaussian_blur=self._gaussian_blur,
                        pass_mask=self._pass_mask,
                    )
                    for _ in range(self._num_views)
                ], axis=0)   # [num_views, H, W, C]
            elif self._preprocess_type == "lejepa_masking":
                views = tf.stack([
                    lejepa_masking_augment_single(
                        image_f,
                        image_size=self._image_size,
                        num_blocks=self._masking_num_blocks,
                        scale_range=self._masking_scale_range,
                        aspect_ratio_range=self._masking_aspect_ratio_range,
                        pass_mask=self._pass_mask,
                    )
                    for _ in range(self._num_views)
                ], axis=0)   # [num_views, H, W, C]
            else:
                raise ValueError(
                    f"Unknown preprocess_type: {self._preprocess_type!r}. "
                    "Supported: 'simclr', 'lejepa_masking'.")
        else:
            single_view = eval_preprocess(image_f, image_size=self._image_size,
                                          pass_mask=self._pass_mask)
            views = tf.stack([single_view] * self._num_views,
                             axis=0)  # [num_views, H, W, C]

        return raw_image, views, label

    # -- Iteration --

    def __iter__(self) -> Iterator[DataBatch]:
        for raw, views, labels in self._ds:
            yield (raw.numpy(), views.numpy(), labels.numpy())

    def __len__(self) -> int:
        if self._drop_remainder:
            return self._dataset_size // self._batch_size
        return math.ceil(self._dataset_size / self._batch_size)


# ---------------------------------------------------------------------------
# Public API — drop-in replacements for the torch-based versions
# ---------------------------------------------------------------------------

def make_cifar10_dataset(
    split: str,
    batch_size: int,
    training: bool,
    data_dir: Optional[str] = None,
    shuffle_buffer: int = 50_000,
    seed: Optional[int] = None,
    drop_remainder: Optional[bool] = None,
    cache: bool = True,
    preprocess_type: str = "simclr",
    num_views: int = 2,
    masking_num_blocks: int = 4,
    masking_scale_range: tuple[float, float] = (0.15, 0.2),
    masking_aspect_ratio_range: tuple[float, float] = (0.75, 1.5),
    pass_mask: bool = False,
    # Torch-era arguments accepted but ignored for compatibility.
    num_workers: int = 0,
    pin_memory: bool = False,
    persistent_workers: Optional[bool] = None,
    download: bool = True,
) -> TFDSLoader:
    """Create a CIFAR-10 data loader yielding NumPy batches for JAX."""
    del shuffle_buffer, pin_memory, persistent_workers, download
    if split not in {"train", "test", "eval"}:
        raise ValueError(
            f"split must be 'train', 'test', or 'eval', got {split!r}.")
    if preprocess_type not in {"simclr", "lejepa_masking"}:
        raise ValueError(
            f"Unknown preprocess_type: {preprocess_type!r}. "
            "Supported: 'simclr', 'lejepa_masking'.")

    return TFDSLoader(
        split=split,
        batch_size=batch_size,
        training=training,
        num_views=num_views,
        preprocess_type=preprocess_type,
        masking_num_blocks=masking_num_blocks,
        masking_scale_range=masking_scale_range,
        masking_aspect_ratio_range=masking_aspect_ratio_range,
        pass_mask=pass_mask,
        seed=seed,
        data_dir=data_dir,
        drop_remainder=drop_remainder,
        cache=cache,
    )


def load_dataset(
    dataset_name: str,
    split: str,
    is_training: bool,
    batch_size: int,
    seed: int,
    **kwargs,
):
    """Load a dataset and return a JAX-friendly DataLoader wrapper."""
    if dataset_name != "cifar10":
        raise ValueError(
            f"Unknown dataset {dataset_name}. Only 'cifar10' is supported.")
    if split not in ("train", "test"):
        raise ValueError(
            f"Unknown split {split}. Supported splits: ('train', 'test')")
    return make_cifar10_dataset(
        split=split,
        batch_size=batch_size,
        training=is_training,
        seed=seed,
        **kwargs,
    )


print("data loading defined.")


## Part 7 — Linear-probe evaluation (`classification`)

Trains a frozen-feature linear/MLP probe to measure representation quality.
`create_feature_extractor` builds a function that reads a given feature
source (`"latent"`, `"recognition"`, `"trunk"`, `"concatenated"`) off a
trained model — this is the piece that dispatches on `model.rpm_type`, so it
works uniformly for Gaussian RPM, Categorical RPM, and SimCLR.

**Known remaining limitation (not fixed here):** for the Gaussian RPM,
`"concatenated"` still doesn't work — the post-combination latent mean has
shape `(batch, dim)` (no per-factor axis; it's already summed over factors)
while the trunk is `(n_factors, batch, dim)`, so concatenating them fails.
This is a deeper shape-convention question than the two fixes above (which
were unambiguous), so it's left as-is rather than guessed at. Both
experiments below default to `"trunk"`, which works correctly and is
unaffected by this.


In [ ]:
FeatureSource = str    # one of "latent", "recognition", "trunk", "concatenated"
PosteriorMeanFn = Any


class ProbeClassifier(nn.Module):
    """Lightweight classification head used to probe learned features."""

    probe_type: str    # "linear" or "mlp"
    num_classes: int
    hidden_dims: tuple[int, ...] = (512, 256)
    dropout_rate: float = 0.1

    @nn.compact
    def __call__(self, x: jax.Array, train: bool) -> jax.Array:
        # Linear probe: a single affine map, no hidden layers / dropout.
        if self.probe_type == "linear":
            return nn.Dense(self.num_classes, name="linear_head")(x)

        # MLP probe: dropout is disabled when ``train`` is False.
        for i, hidden_dim in enumerate(self.hidden_dims):
            x = nn.Dense(hidden_dim, name=f"mlp_dense_{i}")(x)
            x = nn.relu(x)
            x = nn.Dropout(rate=self.dropout_rate,
                           name=f"mlp_dropout_{i}")(x, deterministic=not train)
        return nn.Dense(self.num_classes, name="mlp_head")(x)


def create_feature_extractor(
        model: nn.Module,
        feature_source: FeatureSource = "latent") -> PosteriorMeanFn:
    """Build a jitted function returning per-image features from a model.

    All returned functions produce arrays with a leading **view** axis so
    that ``compute_features_and_labels`` can uniformly index ``feats[0]``
    to select the first view and obtain a ``(N, feature_dim)`` array.

    Feature sources that already carry a view axis (``"trunk"`` and SimCLR
    ``"latent"``) are returned as-is.  Sources that aggregate across views
    (Gaussian/Categorical RPM ``"latent"``, ``"recognition"``) are wrapped
    with ``jnp.expand_dims(..., axis=0)`` to add a dummy leading axis.
    """
    if model.rpm_type == "categorical":
        assert feature_source in ("latent", "recognition"), (
            f"Unsupported feature_source {feature_source} for categorical RPM. "
        )
    elif model.rpm_type == "simclr":
        assert feature_source in ("latent", "trunk", "concatenated"), (
            f"Unsupported feature_source {feature_source} for a SimCLR "
            "encoder (there is no separate recognition factor).")
    is_categorical = model.rpm_type == "categorical"
    is_simclr = model.rpm_type == "simclr"

    if feature_source == "latent":

        @jax.jit
        def latent_posterior_mean_fn(params: Params,
                                     batch_views: jax.Array) -> jax.Array:
            main_output, _ = model.apply({"params": params}, batch_views)
            if is_simclr:
                # SimCLREncoder's primary output *is* the latent (projection)
                # feature, already shaped like aux["trunk"] (view axis first).
                return main_output
            elif is_categorical:
                # variational.probs is (N, K) — no view axis; add one so
                # compute_features_and_labels can index [0].
                return jnp.expand_dims(main_output["variational"].probs,
                                       axis=0)
            else:
                # FIX: the original read `main_output["variational"].mean`,
                # but GaussianNatParams/DiagGaussianNatParams have no `.mean`
                # attribute -- only precision/precision_weighted_mean. The
                # actual mean lives in the separately-computed
                # `variational_mean` (a GaussianMeanParams/
                # DiagGaussianMeanParams, which does have `.mean`).
                #
                # variational_mean.mean is (N, dim_latent) — no view axis;
                # add one so compute_features_and_labels can index [0].
                return jnp.expand_dims(
                    main_output["variational_mean"].mean, axis=0)

        return latent_posterior_mean_fn
    elif feature_source == "recognition":

        @jax.jit
        def recognition_posterior_mean_fn(params: Params,
                                          batch_views: jax.Array) -> jax.Array:
            rpm_outputs, _ = model.apply({"params": params}, batch_views)
            if is_categorical:
                # factors[0].probs is (N, K) — add view axis.
                return jnp.expand_dims(rpm_outputs["factors"][0].probs,
                                       axis=0)
            else:
                # recognition[0] selects factor 0; convert to mean params.
                # The result is (N, dim_latent) — add view axis.
                rec_mean = diag_mean_params(rpm_outputs["recognition"][0]).mean
                return jnp.expand_dims(rec_mean, axis=0)

        return recognition_posterior_mean_fn
    elif feature_source == "trunk":

        @jax.jit
        def trunk_posterior_mean_fn(params: Params,
                                    batch_views: jax.Array) -> jax.Array:
            _, aux = model.apply({"params": params}, batch_views)
            return aux["trunk"]

        return trunk_posterior_mean_fn
    elif feature_source == "concatenated":

        @jax.jit
        def concatenated_posterior_mean_fn(params: Params,
                                           batch_views: jax.Array) -> jax.Array:
            # FIX: the original did `_, aux = model.apply(...)`, discarding
            # the tuple element that actually holds the latent mean, then
            # tried `aux["variational"]` -- but `aux` never has that key (it
            # only ever holds {"trunk", "projection"}). Now both elements are
            # kept and the latent is read from the right one.
            main_output, aux = model.apply({"params": params}, batch_views)
            trunk = aux["trunk"]  # (J, N, trunk_dim)
            if is_simclr:
                latent = main_output  # (J, N, projection_dim)
            else:
                # (N, dim_latent) — broadcast to match trunk's view dim.
                latent_2d = main_output["variational_mean"].mean
                latent = jnp.broadcast_to(
                    latent_2d[jnp.newaxis],
                    (trunk.shape[0],) + latent_2d.shape)
            return jnp.concatenate([latent, trunk], axis=-1)

        return concatenated_posterior_mean_fn
    else:
        raise ValueError(
            f"Unsupported feature_source {feature_source}. "
            f"Supported sources: ('latent', 'recognition', 'trunk', 'concatenated')"
        )


def compute_features_and_labels(
    posterior_mean_fn: PosteriorMeanFn,
    params: Params,
    data_loader,
) -> tuple[jax.Array, jax.Array]:
    """Extract posterior-mean features for every image, in mini-batches."""
    iter_ds = iter(data_loader)
    features = []
    labels_list = []
    for (_, views, label) in iter_ds:
        feats = posterior_mean_fn(params, views)
        # We assume that we do not have randomness during eval.
        feats = feats[0]
        features.append(feats)
        labels_list.append(label)
    features = jnp.concatenate(features, axis=0)
    labels = jnp.concatenate(labels_list, axis=0)
    return features, labels


def run_probe(
    model: nn.Module,
    model_params: Params,
    probe_model: nn.Module,
    train_dataloader,
    test_dataloader,
    init_rng: jax.Array,
    output_dir: Path | None,
    lr: float = 0.01,
    weight_decay: float = 0.0,
    num_epochs: int = 20,
    batch_size: int = 256,
    probe_type: str = "linear",
    mlp_hidden_dims: Sequence[int] = (512, 256),
    mlp_dropout: float = 0.1,
    step: int = 0,
    feature_source: FeatureSource = "latent",
) -> float:
    """Train a probe classifier on frozen features, return best val acc."""
    posterior_mean_fn = create_feature_extractor(model,
                                                 feature_source=feature_source)
    probe_tag = f"{probe_type}-probe"
    probe_dir_name = f"{probe_type}_probe"
    hidden_dims = tuple(int(d) for d in mlp_hidden_dims)

    print("Pre-computing features for training set...")
    train_feats, train_labels = compute_features_and_labels(
        posterior_mean_fn=posterior_mean_fn,
        params=model_params,
        data_loader=train_dataloader,
    )
    train_feats = jax.lax.stop_gradient(train_feats)

    print("Pre-computing features for validation set...")
    val_feats, val_labels = compute_features_and_labels(
        posterior_mean_fn=posterior_mean_fn,
        params=model_params,
        data_loader=test_dataloader,
    )
    val_feats = jax.lax.stop_gradient(val_feats)

    optimizer = optax.sgd(
        learning_rate=lr,
        momentum=0.9,
        nesterov=True,
    )
    if weight_decay > 0:
        optimizer = optax.chain(optax.add_decayed_weights(weight_decay),
                                optimizer)

    init_rngs = {"params": init_rng, "dropout": init_rng}

    if feature_source == "latent":
        probe_feat_dim = model.latent_dim
    elif feature_source == "recognition":
        probe_feat_dim = model.latent_dim
    elif feature_source == "trunk":
        probe_feat_dim = model.trunk_dim
    elif feature_source == "concatenated":
        probe_feat_dim = model.latent_dim + model.trunk_dim
    else:
        raise ValueError(f"Unknown feature source: {feature_source}")

    dummy_x = jnp.zeros((1, probe_feat_dim), dtype=jnp.float32)
    init_probe_vars = probe_model.init(init_rngs, dummy_x, train=True)
    probe_params = init_probe_vars["params"]

    opt_state = optimizer.init(probe_params)

    @jax.jit
    def train_step(
        params,
        opt_state,
        features,
        labels,
        dropout_key,
    ):

        def loss_fn(curr_params):
            logits = probe_model.apply(
                {"params": curr_params},
                features,
                train=True,
                rngs={"dropout": dropout_key},
            )
            loss = jnp.mean(
                optax.softmax_cross_entropy_with_integer_labels(logits, labels))
            acc = jnp.mean((jnp.argmax(logits,
                                       axis=-1) == labels).astype(jnp.float32))
            return loss, acc

        (loss, acc), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
        updates, new_opt_state = optimizer.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        return new_params, new_opt_state, loss, acc

    @jax.jit
    def eval_step(params, features, labels):
        logits = probe_model.apply({"params": params}, features, train=False)
        loss = jnp.mean(
            optax.softmax_cross_entropy_with_integer_labels(logits, labels))
        acc = jnp.mean((jnp.argmax(logits,
                                   axis=-1) == labels).astype(jnp.float32))
        return loss, acc

    best_val_acc = 0.0
    best_epoch = 0
    rng = jax.random.PRNGKey(1)
    n_train = train_feats.shape[0]
    n_val = val_feats.shape[0]
    for epoch in range(num_epochs):
        train_correct = 0
        train_total = 0
        train_loss_sum = 0.0

        for i in range(0, n_train, batch_size):
            batch_x = train_feats[i:i + batch_size]
            batch_y = train_labels[i:i + batch_size]
            rng, step_rng = jax.random.split(rng)
            probe_params, opt_state, loss, acc = train_step(
                probe_params, opt_state, batch_x, batch_y, step_rng)

            batch_count = batch_y.shape[0]
            train_loss_sum += float(loss) * batch_count
            train_total += batch_count
            train_correct += int(round(float(acc) * batch_count))

        val_correct = 0
        val_total = 0
        val_loss_sum = 0.0

        for i in range(0, n_val, batch_size):
            batch_x = val_feats[i:i + batch_size]
            batch_y = val_labels[i:i + batch_size]

            loss, acc = eval_step(probe_params, batch_x, batch_y)

            batch_count = batch_y.shape[0]
            val_loss_sum += float(loss) * batch_count
            val_total += batch_count
            val_correct += int(round(float(acc) * batch_count))

        train_acc = train_correct / max(train_total, 1)
        val_acc = val_correct / max(val_total, 1)
        train_loss = train_loss_sum / max(train_total, 1)
        val_loss = val_loss_sum / max(val_total, 1)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            if output_dir is not None:
                probe_ckpt_dir = output_dir / probe_dir_name
                probe_ckpt_dir.mkdir(parents=True, exist_ok=True)
                checkpoint_path = (
                    probe_ckpt_dir /
                    f"best_{probe_type}_probe_step_{step}").resolve()
                payload = {
                    "epoch": int(best_epoch),
                    "val_acc": float(best_val_acc),
                    "probe_type": probe_type,
                    "hidden_dims": tuple(int(v) for v in hidden_dims),
                    "dropout": float(mlp_dropout),
                    "params": serialization.to_state_dict(probe_params),
                }
                checkpointer = ocp.PyTreeCheckpointer()
                checkpointer.save(str(checkpoint_path), payload, force=True)

        print(f"[{probe_tag}] epoch={epoch + 1:03d}/{num_epochs:03d} "
              f"train_loss={train_loss:.5f} val_loss={val_loss:.5f} "
              f"train_acc={train_acc * 100:.2f}% val_acc={val_acc * 100:.2f}% "
              f"best_val_acc={best_val_acc * 100:.2f}%")

    print(
        f"[{probe_tag}] best_val_acc={best_val_acc * 100:.2f}% at epoch {best_epoch}"
    )
    return best_val_acc


print("classification defined.")


## Part 8 — Training-step builders and checkpointing (`train_helpers`)

`_get_loss_fn` dispatches on `model.rpm_type` to pick the right free-energy
function — this is the one piece of glue that makes the *same* generic
training loop below work for Gaussian RPM, Categorical RPM, and SimCLR.


In [ ]:
@struct.dataclass
class TrainState:
    params: Params
    opt_state: optax.OptState
    rng: jax.Array
    step: int


def _get_loss_fn(model):
    """Return the (params, model, batch, rng, beta) -> (free_energy, metrics)
    function for ``model``. Despite the name, this covers non-RPM models too
    (e.g. SimCLR): "free_energy" here just means "the quantity being
    maximized", matching create_train_step's ``loss = -free_energy``.
    """
    if model.rpm_type == "categorical":
        return free_energy_categorical
    elif model.rpm_type == "gaussian":
        return free_energy
    elif model.rpm_type == "simclr":
        return nt_xent_loss
    else:
        raise ValueError(f"Unknown RPM type: {model.rpm_type}")


def create_train_step(
    model,
    optimizer: optax.GradientTransformation,
    beta_schedule: float | optax.Schedule = 1.0,
):
    """Build a jitted single optimization step that maximizes the free energy."""
    free_energy_fn = _get_loss_fn(model)
    beta_schedule = (beta_schedule if callable(beta_schedule) else
                     optax.constant_schedule(beta_schedule))

    @jax.jit
    def train_step(
            state: TrainState,
            batch_views: jax.Array) -> tuple[TrainState, dict[str, Array]]:
        rng, loss_rng = jax.random.split(state.rng)
        beta_t = beta_schedule(state.step)

        def loss_fn(params, rng):
            vars_dict = {"params": params}
            free_energy_val, aux_terms = free_energy_fn(
                vars_dict,
                model,
                batch_views,
                rng=rng,
                beta=beta_t,
            )
            # We maximize the free energy, so minimize its negative.
            loss = -free_energy_val
            return loss, (free_energy_val, aux_terms)

        (loss, (free_energy_val,
                aux_terms)), grads = value_and_grad(loss_fn,
                                                    has_aux=True)(state.params,
                                                                  loss_rng)

        # Raw global gradient norm (before any clipping). A sudden spike here is
        # the early-warning signal for the divergence that blows a healthy run
        # into the collapsed (free energy -> 0) state.
        grad_norm = optax.global_norm(grads)

        updates, new_opt_state = optimizer.update(grads, state.opt_state,
                                                  state.params)
        new_params = optax.apply_updates(state.params, updates)

        new_state = state.replace(
            params=new_params,
            opt_state=new_opt_state,
            rng=rng,
            step=state.step + 1,
        )

        metrics = {
            "loss": loss,
            "free_energy": free_energy_val,
            "grad_norm": grad_norm,
            "beta": beta_t,
        }
        metrics.update(aux_terms)

        return new_state, metrics

    return train_step


def create_eval_step(model, beta_schedule: float | optax.Schedule = 1.0):
    """Build a jitted function returning the free energy for a batch."""
    free_energy_fn = _get_loss_fn(model)

    beta_schedule = (beta_schedule if callable(beta_schedule) else
                     optax.constant_schedule(beta_schedule))

    @jax.jit
    def eval_step(params: Params, batch_views: jax.Array, step: int,
                  rng: Array) -> Array:
        beta_t = beta_schedule(step)
        free_energy_val, _ = free_energy_fn({"params": params},
                                            model,
                                            batch_views,
                                            rng=rng,
                                            beta=beta_t)
        return free_energy_val

    return eval_step


def _checkpoint_metadata(history: dict, best_val_fe: float) -> dict:
    """Build a JSON-serializable metadata payload for Orbax checkpoints."""
    return {
        "history": history,
        "best_val_fe": float(best_val_fe),
    }


def save_rpm_params_checkpoint(
    ckpt_manager: ocp.CheckpointManager,
    step: int,
    params: Params,
    history: dict,
    best_val_fe: float,
) -> None:
    """Save model params (not full training state) plus lightweight metadata."""
    ckpt_manager.save(
        int(step),
        args=ocp.args.Composite(
            params=ocp.args.StandardSave(params),
            metadata=ocp.args.JsonSave(
                _checkpoint_metadata(history, best_val_fe)),
        ),
    )


def save_checkpoint(
    ckpt_manager: ocp.CheckpointManager,
    current_state: TrainState,
    current_history: dict,
    current_best_val_fe: float,
) -> None:
    save_rpm_params_checkpoint(
        ckpt_manager=ckpt_manager,
        step=int(current_state.step),
        params=current_state.params,
        history=current_history,
        best_val_fe=current_best_val_fe,
    )


def plot_training_metrics(
    history,
    *,
    val_every=None,
    probe_every=None,
    title=None,
):
    """Plot train free energy, val free energy, and best probe accuracy."""

    def _infer_interval(total_steps, count):
        if count <= 0 or total_steps <= 0:
            return None
        interval = int(round(total_steps / count))
        return interval if interval > 0 else None

    steps = np.asarray(history.get("step", []), dtype=int)

    if "free_energy" in history and len(history["free_energy"]) > 0:
        free_energy_hist = np.asarray(history["free_energy"][1:], dtype=float)
    else:
        losses = np.asarray(history.get("loss", [])[1:], dtype=float)
        free_energy_hist = -losses

    if steps.size == 0:
        steps = np.arange(1, free_energy_hist.size + 1)

    val_free_energy = np.asarray(history.get("val_free_energy", []),
                                 dtype=float)
    probe_val_acc = np.asarray(history.get("probe_val_acc", []), dtype=float)

    val_steps = np.array([], dtype=int)
    if val_free_energy.size > 0:
        interval = val_every
        if interval is None and steps.size > 0:
            interval = _infer_interval(int(steps[-1]),
                                       int(val_free_energy.size))
        if interval is None:
            interval = 1
        val_steps = np.arange(1, val_free_energy.size + 1) * interval

    probe_steps = np.array([], dtype=int)
    if probe_val_acc.size > 0:
        interval = probe_every
        if interval is None and steps.size > 0:
            interval = _infer_interval(int(steps[-1]), int(probe_val_acc.size))
        if interval is None:
            interval = 1
        probe_steps = np.arange(0, probe_val_acc.size) * interval

    acc_label = "best val probe acc"
    if probe_val_acc.size > 0 and probe_val_acc.max() <= 1.0:
        probe_val_acc = 100.0 * probe_val_acc
        acc_label = "best val probe acc (%)"

    fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    axes[0].plot(steps[1:], free_energy_hist, label="train free energy")
    if val_free_energy.size > 0:
        axes[0].plot(val_steps, val_free_energy, label="val free energy")
    axes[0].set_ylabel("free energy")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    if probe_val_acc.size > 0:
        axes[1].plot(probe_steps, probe_val_acc, label=acc_label)
    else:
        axes[1].text(
            0.5,
            0.5,
            "No probe results logged",
            ha="center",
            va="center",
            transform=axes[1].transAxes,
        )
    axes[1].set_ylabel(acc_label)
    axes[1].set_xlabel("training step")
    axes[1].grid(True, alpha=0.3)

    if title:
        fig.suptitle(title)

    fig.tight_layout()
    return fig, axes


print("train_helpers defined.")


## Part 9 — Generic training loop (`train_rpm`)

This is the same function for both experiments: it dispatches its loss via
`_get_loss_fn(model)` (Part 8), so passing a `GaussianRPM` or a
`SimCLREncoder` in the `model=` argument is the only thing that changes
between the two experiments below.


In [ ]:
def train_rpm(
    model,
    train_dataloader,
    eval_train_dataloader,
    eval_test_dataloader,
    optimizer_name,
    probe_type,
    output_dir: Path = Path("outputs"),
    learning_rate: float | optax.Schedule = 1e-3,
    weight_decay: float = 0.05,
    grad_clip_norm: float | None = 1.0,
    batch_size: int = 64,
    num_steps: int = 1000,
    log_every: int = 100,
    val_every: int = 500,
    val_steps: int = 10,
    probe_every: int = 500,
    probe_epochs: int = 30,
    probe_lr: float = 0.1,
    seed: int = 42,
    beta: float | optax.Schedule = 1.0,
    probe_feature_source: FeatureSource = "latent",
    checkpoint_every: int = 10000,
    save_best: bool = True,
    max_checkpoints_to_keep: int = 3,
) -> tuple[TrainState, dict]:
    """Train ``model`` and return the final state and a metrics history."""
    output_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir = (output_dir / "checkpoints").absolute()
    best_ckpt_dir = (output_dir / "best_checkpoint").absolute()
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt_dir.mkdir(parents=True, exist_ok=True)

    rng = jax.random.PRNGKey(seed)
    rng, init_rng = jax.random.split(rng)

    _, init_batch, _ = next(iter(train_dataloader))
    variables = model.init(init_rng, init_batch)
    lr_schedule = (learning_rate if callable(learning_rate) else
                   optax.constant_schedule(learning_rate))
    tx = getattr(optax, optimizer_name)(learning_rate=lr_schedule,
                                        weight_decay=weight_decay)
    if grad_clip_norm is not None:
        tx = optax.chain(optax.clip_by_global_norm(grad_clip_norm), tx)
    opt_state = tx.init(variables["params"])
    state = TrainState(
        params=variables["params"],
        opt_state=opt_state,
        rng=rng,
        step=0,
    )
    beta_schedule = beta if callable(beta) else optax.constant_schedule(beta)
    train_step_fn = create_train_step(model=model,
                                      optimizer=tx,
                                      beta_schedule=beta_schedule)
    eval_step_fn = create_eval_step(model=model, beta_schedule=beta_schedule)

    history = {
        "step": [],
        "loss": [],
        "lr": [],
        "grad_norm": [],
        "beta": [],
        "val_free_energy": [],
        "probe_val_acc": [],
        "neg_kl_qp": [],
        "neg_kl_qf": [],
        "neg_log_gamma": [],
        "q_z_mean_std": [],
        "q_z_var_mean": [],
        "entropy": [],
        "prior_xent": [],
        "factors_xent": [],
        "denom_xent": [],
    }
    # FIX: the original initialized `best_val_fe = float("inf")` and saved a
    # new "best" checkpoint whenever `mean_val_fe < best_val_fe` -- but free
    # energy is *maximized* during training (verified empirically: it climbs
    # steadily over training steps), so tracking a minimum silently saved the
    # *worst* validation checkpoint as "best". Track the maximum instead.
    best_val_fe = float("-inf")
    main_options = ocp.CheckpointManagerOptions(
        max_to_keep=max_checkpoints_to_keep,
        create=True,
    )
    best_options = ocp.CheckpointManagerOptions(
        max_to_keep=1,
        create=True,
    )
    print(f"Starting training for {num_steps} steps, ")
    start_time = time.time()
    probe_model = ProbeClassifier(
        probe_type=probe_type,
        num_classes=train_dataloader.num_classes,
    )
    with (
            ocp.CheckpointManager(ckpt_dir,
                                  options=main_options) as ckpt_manager,
            ocp.CheckpointManager(best_ckpt_dir, options=best_options) as
            best_ckpt_manager,
    ):
        # Random initialization probe.
        accuracy = run_probe(
            model=model,
            model_params=state.params,
            probe_model=probe_model,
            train_dataloader=eval_train_dataloader,
            test_dataloader=eval_test_dataloader,
            output_dir=output_dir,
            num_epochs=probe_epochs,
            lr=probe_lr,
            weight_decay=0.0,
            batch_size=batch_size,
            step=int(state.step),
            feature_source=probe_feature_source,
            init_rng=init_rng,
        )
        print(f"Probe Validation Accuracy: {accuracy * 100:.2f}%")
        history["probe_val_acc"].append(float(accuracy))

        if checkpoint_every > 0:
            save_checkpoint(ckpt_manager=ckpt_manager,
                            current_state=state,
                            current_history=history,
                            current_best_val_fe=best_val_fe)

        iter_train_ds = iter(train_dataloader)

        for step in range(num_steps):
            state_rng, _ = jax.random.split(state.rng)
            state = state.replace(rng=state_rng)

            # Get the batch, cycling to a new epoch once the current one is
            # exhausted (train_dataloader only iterates for one epoch).
            # Re-iterating draws a fresh shuffle, not a repeat of the same
            # order: the DataLoader is built with shuffle=True and a seeded
            # torch.Generator that it reuses across iterations, and that
            # generator's internal state advances on every draw -- so each
            # new epoch's permutation differs from the last, even though the
            # run as a whole is still reproducible from the original seed.
            try:
                _, batch_views, _ = next(iter_train_ds)
            except StopIteration:
                iter_train_ds = iter(train_dataloader)
                _, batch_views, _ = next(iter_train_ds)

            state, metrics = train_step_fn(state, batch_views)

            if (step + 1) % log_every == 0 or step == 0:
                elapsed = time.time() - start_time
                current_lr = float(lr_schedule(int(state.step)))

                print(f"Step {step + 1}/{num_steps} | "
                      f"Loss: {float(metrics['loss']):.4f} | "
                      f"Free Energy: {float(metrics['free_energy']):.4f} | "
                      f"grad_norm: {float(metrics['grad_norm']):.4f} | "
                      f"beta: {float(metrics['beta']):.4f} | "
                      f"LR: {current_lr:.5f} | "
                      f"Elapsed: {elapsed:.4f}")
                history["step"].append(step + 1)
                history["lr"].append(current_lr)
                for key, value in metrics.items():
                    history.setdefault(key, []).append(float(value))

            if (step + 1) % val_every == 0:
                val_fes = []
                eval_ds = iter(eval_test_dataloader)
                eval_rng, state_rng = jax.random.split(state.rng)
                for _ in range(val_steps):
                    eval_rng, _ = jax.random.split(eval_rng)
                    _, v_views, _ = next(eval_ds)
                    val_fe = eval_step_fn(state.params, v_views, state.step,
                                          eval_rng)
                    val_fes.append(float(val_fe))

                mean_val_fe = float(jnp.mean(jnp.array(val_fes)))
                print(f"Validation Free Energy: {mean_val_fe:.4f}")
                history["val_free_energy"].append(mean_val_fe)

                # FIX: `>` instead of the original `<` -- see best_val_fe note above.
                if save_best and mean_val_fe > best_val_fe:
                    best_val_fe = mean_val_fe
                    print(
                        f"New best validation free energy: {best_val_fe:.4f}. "
                        "Saving best checkpoint...")
                    save_checkpoint(
                        ckpt_manager=best_ckpt_manager,
                        current_state=state,
                        current_history=history,
                        current_best_val_fe=best_val_fe)

            if (step + 1) % probe_every == 0:
                print("Running probe evaluation")
                accuracy = run_probe(
                    model=model,
                    model_params=state.params,
                    probe_model=probe_model,
                    train_dataloader=eval_train_dataloader,
                    test_dataloader=eval_test_dataloader,
                    output_dir=output_dir,
                    num_epochs=probe_epochs,
                    lr=probe_lr,
                    weight_decay=0.0,
                    batch_size=batch_size,
                    step=int(state.step),
                    feature_source=probe_feature_source,
                    init_rng=state.rng,
                )
                print(f"Probe Validation Accuracy: {accuracy * 100:.2f}%")
                history["probe_val_acc"].append(float(accuracy))

            if checkpoint_every > 0 and ((step + 1) % checkpoint_every == 0 or
                                         (step + 1) == num_steps):
                print(f"Saving checkpoint at step {step + 1}...")
                save_checkpoint(ckpt_manager=ckpt_manager,
                                current_state=state,
                                current_history=history,
                                current_best_val_fe=best_val_fe)

        print("Saving final checkpoint...")
        save_checkpoint(ckpt_manager=ckpt_manager,
                        current_state=state,
                        current_history=history,
                        current_best_val_fe=best_val_fe)

    return state, history


print("train_rpm defined.")


## Visualize data views

Load a small batch from CIFAR-10 with both preprocessing strategies and
display a few sample images alongside their generated views.


In [ ]:
NUM_SAMPLES_TO_SHOW = 6
NUM_VIEWS_VIS = 4

# --- Load a small batch with each strategy ---
_vis_simclr = load_dataset(
    dataset_name="cifar10", split="train", batch_size=NUM_SAMPLES_TO_SHOW,
    seed=0, preprocess_type="simclr", num_views=NUM_VIEWS_VIS, is_training=True,
)
_vis_masking = load_dataset(
    dataset_name="cifar10", split="train", batch_size=NUM_SAMPLES_TO_SHOW,
    seed=0, preprocess_type="lejepa_masking", num_views=NUM_VIEWS_VIS,
    is_training=True,
)

_raw_simclr, _views_simclr, _labels_simclr = next(iter(_vis_simclr))
_raw_masking, _views_masking, _labels_masking = next(iter(_vis_masking))

def _unnormalize(view: np.ndarray) -> np.ndarray:
    """Undo per-channel normalization and clip to [0, 1] for display."""
    return np.clip(view * CIFAR10_STD + CIFAR10_MEAN, 0.0, 1.0)

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

for strategy_name, raw_imgs, views_batch, labels in [
    ("SimCLR", _raw_simclr, _views_simclr, _labels_simclr),
    ("LeJEPA masking", _raw_masking, _views_masking, _labels_masking),
]:
    n_samples = raw_imgs.shape[0]
    n_views = views_batch.shape[1]
    fig, axes = plt.subplots(
        n_samples, 1 + n_views,
        figsize=(2.2 * (1 + n_views), 2.2 * n_samples),
    )
    fig.suptitle(f"Views: {strategy_name}", fontsize=14, fontweight="bold", y=1.01)

    for i in range(n_samples):
        label_str = CIFAR10_CLASSES[labels[i]]
        # Original image
        axes[i, 0].imshow(raw_imgs[i])
        axes[i, 0].set_title(f"Original\n({label_str})", fontsize=9)
        axes[i, 0].axis("off")

        # Augmented / masked views
        for v in range(n_views):
            view_img = _unnormalize(views_batch[i, v])
            axes[i, v + 1].imshow(view_img)
            axes[i, v + 1].set_title(f"View {v + 1}", fontsize=9)
            axes[i, v + 1].axis("off")

    plt.tight_layout()
    plt.show()

del _vis_simclr, _vis_masking
del _raw_simclr, _views_simclr, _labels_simclr
del _raw_masking, _views_masking, _labels_masking


In [ ]:
raise ValueError('ok')

---
# Experiment 1 — Gaussian RPM on CIFAR-10

A ResNet-18-style backbone, `"constrained"` auxiliary construction, diagonal
precision. Mirrors the original `examples/cifar10.py` script's defaults,
just with plain Python variables instead of `absl` flags.

**Defaults below are a quick smoke run** (`num_steps=2000`) so you can
confirm everything works end-to-end before committing to a long run — bump
`NUM_STEPS` (and the schedule-related steps below it) up for a real result.
On a Colab T4 GPU this backbone does roughly 5-10 steps/sec; on CPU expect
well under 1 step/sec, so keep steps low unless you have a GPU runtime.


In [ ]:
def set_global_seed(seed: int) -> None:
    """Set seeds for reproducibility across common Python ML libraries."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


def get_resnet18_cfg(projection_features):
    return {
        "stage_sizes": (2, 2, 2, 2),
        "stage_widths": (64, 128, 256, 512),
        "stem_width": 64,
        "stem_kernel_size": (3, 3),
        "stem_stride": 1,
        "use_max_pool": False,
        "block_type": "basic",
        "projection_features": projection_features,
    }


# --- Config (mirrors examples/cifar10.py's flag defaults) ---
SEED = 42
BATCH_SIZE = 128
EVAL_BATCH_SIZE = 256
NUM_VIEWS = 8
NUM_WORKERS = 0            # Colab: keep at 0: DataLoader workers + notebooks
                            # sometimes deadlock. Increase only in a plain script.
LATENT_DIM = 64
CACHE = False
PREPROCESS_TYPE = "simclr"  # "simclr" or "lejepa_masking"
# PREPROCESS_TYPE = "lejepa_masking"  # "simclr" or "lejepa_masking"

NUM_STEPS = 100_000            # quick smoke run; original default is 100_000
PROBE_EVERY = 10000
VAL_EVERY = 5000
PROBE_EPOCHS = 10
PROBE_LR = 0.1
BETA_INIT = 0.1
BETA_FINAL = 1.0
USE_BETA_SCHEDULE = True
BETA_STEPS = min(20_000, NUM_STEPS)
USE_LR_SCHEDULE = False
CONSTANT_LR = 1e-4
INIT_LR = 1e-4
PEAK_LR = 0.1
MIN_LR = 1e-5
WARMUP_STEPS = 10_000
WEIGHT_DECAY = 1e-6
GRAD_CLIP_NORM = 1.0
PROBE_FEATURE_SOURCE = "latent"
PROJECTION_FEATURES = (2048,)
OPTIMIZER_NAME = "adamw"
PROBE_TYPE = "linear"
LOG_EVERY = 2000
VAL_STEPS = 10
CHECKPOINT_EVERY = 20_000
AUXILIARY_METHOD = "constrained"
SHARE_RECOGNITION = True
PRECISION_TYPE = "diag"
FIX_PRIOR = True
N_SAMPLES = 100
SAVE_BEST = True
MAX_CHECKPOINTS_TO_KEEP = 10

set_global_seed(SEED)
print(sys.executable if hasattr(sys, "executable") else "n/a")
print(jax.devices())


In [ ]:
# --- Data ---
rpm_train_cifar10 = load_dataset(
    dataset_name="cifar10", split="train", batch_size=BATCH_SIZE, seed=SEED,
    cache=CACHE, preprocess_type=PREPROCESS_TYPE, num_views=NUM_VIEWS,
    num_workers=NUM_WORKERS, is_training=True,
)
rpm_eval_train_cifar10 = load_dataset(
    dataset_name="cifar10", split="train", batch_size=EVAL_BATCH_SIZE, seed=SEED,
    cache=CACHE, preprocess_type=PREPROCESS_TYPE, num_views=NUM_VIEWS,
    num_workers=NUM_WORKERS, is_training=False,
)
rpm_eval_test_cifar10 = load_dataset(
    dataset_name="cifar10", split="test", batch_size=EVAL_BATCH_SIZE, seed=SEED,
    cache=CACHE, preprocess_type=PREPROCESS_TYPE, num_views=NUM_VIEWS,
    num_workers=NUM_WORKERS, is_training=False,
)

_train_batch = next(iter(rpm_train_cifar10))
_train_raw, _train_views, _train_labels = _train_batch
print("Train raw images:", _train_raw.shape, _train_raw.dtype)
print("Train views:", _train_views.shape, _train_views.dtype)
print("Train labels:", _train_labels.shape, _train_labels.dtype)
print("Train dataset size:", rpm_train_cifar10.dataset_size)
print("Num classes:", rpm_train_cifar10.num_classes)


In [ ]:
# --- Model + schedules ---
if USE_LR_SCHEDULE:
    rpm_lr_schedule = optax.warmup_cosine_decay_schedule(
        init_value=INIT_LR, warmup_steps=WARMUP_STEPS,
        decay_steps=NUM_STEPS, peak_value=PEAK_LR, end_value=MIN_LR,
    )
else:
    rpm_lr_schedule = optax.constant_schedule(value=CONSTANT_LR)

if USE_BETA_SCHEDULE:
    rpm_beta_schedule = optax.linear_schedule(
        init_value=BETA_INIT, end_value=BETA_FINAL,
        transition_steps=BETA_STEPS, transition_begin=0,
    )
else:
    rpm_beta_schedule = optax.constant_schedule(value=BETA_INIT)

gaussian_model = GaussianRPM.create(
    auxiliary_method=AUXILIARY_METHOD,
    n_factors=NUM_VIEWS,
    dim_latent=LATENT_DIM,
    encoder_arch="resnet",
    encoder_params=get_resnet18_cfg(PROJECTION_FEATURES),
    share_recognition=SHARE_RECOGNITION,
    precision_type=PRECISION_TYPE,
    fix_prior=FIX_PRIOR,
    n_samples=N_SAMPLES,
)
print("Gaussian RPM model constructed.")


In [ ]:
# --- Train ---
rpm_output_dir = Path("/tmp/outputs/cifar10_gaussian_rpm") / time.strftime("%Y%m%d-%H%M%S")

rpm_final_state, rpm_history = train_rpm(
    model=gaussian_model,
    train_dataloader=rpm_train_cifar10,
    eval_train_dataloader=rpm_eval_train_cifar10,
    eval_test_dataloader=rpm_eval_test_cifar10,
    optimizer_name=OPTIMIZER_NAME,
    probe_type=PROBE_TYPE,
    output_dir=rpm_output_dir,
    learning_rate=rpm_lr_schedule,
    weight_decay=WEIGHT_DECAY,
    grad_clip_norm=GRAD_CLIP_NORM,
    batch_size=BATCH_SIZE,
    num_steps=NUM_STEPS,
    log_every=LOG_EVERY,
    val_every=VAL_EVERY,
    val_steps=VAL_STEPS,
    probe_every=PROBE_EVERY,
    probe_epochs=PROBE_EPOCHS,
    probe_lr=PROBE_LR,
    seed=SEED,
    beta=rpm_beta_schedule,
    probe_feature_source=PROBE_FEATURE_SOURCE,
    checkpoint_every=CHECKPOINT_EVERY,
    save_best=SAVE_BEST,
    max_checkpoints_to_keep=MAX_CHECKPOINTS_TO_KEEP,
)


In [ ]:
rpm_history['probe_val_acc']


In [ ]:
print('da')

In [ ]:
print('da')

In [ ]:
_, _ = plot_training_metrics(
    rpm_history, val_every=VAL_EVERY, probe_every=PROBE_EVERY,
    title="Gaussian RPM (CIFAR-10, diag) training metrics")
plt.savefig(rpm_output_dir / "training_metrics.png")
plt.show()


---
# Experiment 2 — SimCLR on CIFAR-10

Same ResNet-18 backbone and the same `train_rpm` training loop as
Experiment 1 — only the model (`SimCLREncoder` instead of `GaussianRPM`) and
its loss (NT-Xent instead of free energy, dispatched automatically via
`model.rpm_type`) differ. Standard SimCLR uses `num_views=2`.


In [ ]:
# --- Config (mirrors examples/cifar10_simclr.py's flag defaults) ---
SIMCLR_SEED = 42
SIMCLR_BATCH_SIZE = 128
SIMCLR_EVAL_BATCH_SIZE = 256
SIMCLR_NUM_VIEWS = 8
SIMCLR_NUM_WORKERS = 0
SIMCLR_CACHE = False
SIMCLR_PREPROCESS_TYPE = "simclr"  # "simclr" or "lejepa_masking"
# SIMCLR_PREPROCESS_TYPE = "lejepa_masking"  # "simclr" or "lejepa_masking"

SIMCLR_NUM_STEPS = 100_000     # quick smoke run; original default is 100_000
SIMCLR_PROBE_EVERY = 10_000
SIMCLR_VAL_EVERY = 5000
SIMCLR_PROBE_EPOCHS = 10
SIMCLR_PROBE_LR = 0.1
SIMCLR_USE_LR_SCHEDULE = False
SIMCLR_CONSTANT_LR = 1e-4
SIMCLR_INIT_LR = 1e-4
SIMCLR_PEAK_LR = 0.1
SIMCLR_MIN_LR = 1e-5
SIMCLR_WARMUP_STEPS = 10_000
SIMCLR_WEIGHT_DECAY = 1e-6
SIMCLR_GRAD_CLIP_NORM = 1.0
SIMCLR_PROBE_FEATURE_SOURCE = "trunk"
SIMCLR_PROJECTION_FEATURES = (2048,)
SIMCLR_OPTIMIZER_NAME = "adamw"
SIMCLR_PROBE_TYPE = "linear"
SIMCLR_LOG_EVERY = 2000
SIMCLR_VAL_STEPS = 10
SIMCLR_CHECKPOINT_EVERY = 20_000
SIMCLR_SAVE_BEST = True
SIMCLR_MAX_CHECKPOINTS_TO_KEEP = 10
SIMCLR_TEMPERATURE = 0.5

set_global_seed(SIMCLR_SEED)
print(jax.devices())


In [ ]:
# --- Data ---
simclr_train_cifar10 = load_dataset(
    dataset_name="cifar10", split="train", batch_size=SIMCLR_BATCH_SIZE,
    seed=SIMCLR_SEED, cache=SIMCLR_CACHE, preprocess_type=SIMCLR_PREPROCESS_TYPE,
    num_views=SIMCLR_NUM_VIEWS, num_workers=SIMCLR_NUM_WORKERS, is_training=True,
)
simclr_eval_train_cifar10 = load_dataset(
    dataset_name="cifar10", split="train", batch_size=SIMCLR_EVAL_BATCH_SIZE,
    seed=SIMCLR_SEED, cache=SIMCLR_CACHE, preprocess_type=SIMCLR_PREPROCESS_TYPE,
    num_views=SIMCLR_NUM_VIEWS, num_workers=SIMCLR_NUM_WORKERS, is_training=False,
)
simclr_eval_test_cifar10 = load_dataset(
    dataset_name="cifar10", split="test", batch_size=SIMCLR_EVAL_BATCH_SIZE,
    seed=SIMCLR_SEED, cache=SIMCLR_CACHE, preprocess_type=SIMCLR_PREPROCESS_TYPE,
    num_views=SIMCLR_NUM_VIEWS, num_workers=SIMCLR_NUM_WORKERS, is_training=False,
)

_train_batch = next(iter(simclr_train_cifar10))
_train_raw, _train_views, _train_labels = _train_batch
print("Train raw images:", _train_raw.shape, _train_raw.dtype)
print("Train views:", _train_views.shape, _train_views.dtype)
print("Train labels:", _train_labels.shape, _train_labels.dtype)
print("Train dataset size:", simclr_train_cifar10.dataset_size)
print("Num classes:", simclr_train_cifar10.num_classes)


In [ ]:
# --- Model + schedule ---
if SIMCLR_USE_LR_SCHEDULE:
    simclr_lr_schedule = optax.warmup_cosine_decay_schedule(
        init_value=SIMCLR_INIT_LR, warmup_steps=SIMCLR_WARMUP_STEPS,
        decay_steps=SIMCLR_NUM_STEPS, peak_value=SIMCLR_PEAK_LR,
        end_value=SIMCLR_MIN_LR,
    )
else:
    simclr_lr_schedule = optax.constant_schedule(value=SIMCLR_CONSTANT_LR)

simclr_model = SimCLREncoder(
    n_views=SIMCLR_NUM_VIEWS,
    encoder_arch="resnet",
    encoder_params=get_resnet18_cfg(SIMCLR_PROJECTION_FEATURES),
    temperature=SIMCLR_TEMPERATURE,
)
print("SimCLR model constructed.")


In [ ]:
# --- Train ---
simclr_output_dir = Path("/tmp/outputs/cifar10_simclr") / time.strftime("%Y%m%d-%H%M%S")

simclr_final_state, simclr_history = train_rpm(
    model=simclr_model,
    train_dataloader=simclr_train_cifar10,
    eval_train_dataloader=simclr_eval_train_cifar10,
    eval_test_dataloader=simclr_eval_test_cifar10,
    optimizer_name=SIMCLR_OPTIMIZER_NAME,
    probe_type=SIMCLR_PROBE_TYPE,
    output_dir=simclr_output_dir,
    learning_rate=simclr_lr_schedule,
    weight_decay=SIMCLR_WEIGHT_DECAY,
    grad_clip_norm=SIMCLR_GRAD_CLIP_NORM,
    batch_size=SIMCLR_BATCH_SIZE,
    num_steps=SIMCLR_NUM_STEPS,
    log_every=SIMCLR_LOG_EVERY,
    val_every=SIMCLR_VAL_EVERY,
    val_steps=SIMCLR_VAL_STEPS,
    probe_every=SIMCLR_PROBE_EVERY,
    probe_epochs=SIMCLR_PROBE_EPOCHS,
    probe_lr=SIMCLR_PROBE_LR,
    seed=SIMCLR_SEED,
    probe_feature_source=SIMCLR_PROBE_FEATURE_SOURCE,
    checkpoint_every=SIMCLR_CHECKPOINT_EVERY,
    save_best=SIMCLR_SAVE_BEST,
    max_checkpoints_to_keep=SIMCLR_MAX_CHECKPOINTS_TO_KEEP,
)


In [ ]:
_, _ = plot_training_metrics(
    simclr_history, val_every=SIMCLR_VAL_EVERY, probe_every=SIMCLR_PROBE_EVERY,
    title="SimCLR (CIFAR-10, ResNet-18) training metrics")
plt.savefig(simclr_output_dir / "training_metrics.png")
plt.show()


In [ ]:
print('da')

---
# Latent Space Analysis

Comprehensive comparison of the learned RPM and SimCLR representations.

1. Feature extraction (trunk features + labels for both models)
2. Dimensionality reduction (PCA 2D + explained-variance curve, t-SNE)
3. k-NN accuracy (nearest-neighbor classification without a learned head)
4. Alignment & Uniformity (Wang & Isola, 2020)
5. Inter-class / intra-class structure (centroid distance heatmap + per-class variance)
6. RPM-specific uncertainty analysis (per-class posterior uncertainty, calibration)
7. CKA (Centered Kernel Alignment) cross-method comparison


In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.cluster import KMeans
from scipy.spatial.distance import pdist, squareform

# =====================================================================
# 1. Extract features for both models
# =====================================================================
ANALYSIS_FEATURE_SOURCE = "latent"  # "trunk", "latent", or "concatenated"

# --- RPM features ---
rpm_feat_fn = create_feature_extractor(gaussian_model,
                                        feature_source=ANALYSIS_FEATURE_SOURCE)
rpm_train_feats, rpm_train_labels = compute_features_and_labels(
    rpm_feat_fn, rpm_final_state.params, rpm_eval_train_cifar10)
rpm_test_feats, rpm_test_labels = compute_features_and_labels(
    rpm_feat_fn, rpm_final_state.params, rpm_eval_test_cifar10)

# --- SimCLR features ---
simclr_feat_fn = create_feature_extractor(simclr_model,
                                           feature_source=ANALYSIS_FEATURE_SOURCE)
simclr_train_feats, simclr_train_labels = compute_features_and_labels(
    simclr_feat_fn, simclr_final_state.params, simclr_eval_train_cifar10)
simclr_test_feats, simclr_test_labels = compute_features_and_labels(
    simclr_feat_fn, simclr_final_state.params, simclr_eval_test_cifar10)

# Convert to numpy for sklearn
rpm_train_feats_np = np.array(rpm_train_feats)
rpm_test_feats_np = np.array(rpm_test_feats)
rpm_train_labels_np = np.array(rpm_train_labels)
rpm_test_labels_np = np.array(rpm_test_labels)

simclr_train_feats_np = np.array(simclr_train_feats)
simclr_test_feats_np = np.array(simclr_test_feats)
simclr_train_labels_np = np.array(simclr_train_labels)
simclr_test_labels_np = np.array(simclr_test_labels)

print(f"RPM   train features: {rpm_train_feats_np.shape}")
print(f"RPM   test  features: {rpm_test_feats_np.shape}")
print(f"SimCLR train features: {simclr_train_feats_np.shape}")
print(f"SimCLR test  features: {simclr_test_feats_np.shape}")


In [ ]:
# =====================================================================
# 2. Dimensionality Reduction: PCA (2D + explained variance) and t-SNE
# =====================================================================
N_VIS = 5000  # subsample for t-SNE speed

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

def plot_2d_scatter(feats_2d, labels, title, class_names, ax):
    for c in range(len(class_names)):
        mask = labels == c
        ax.scatter(feats_2d[mask, 0], feats_2d[mask, 1],
                   s=3, alpha=0.4, label=class_names[c])
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# --- PCA 2D ---
for col, (name, feats, labels) in enumerate([
    ("RPM", rpm_test_feats_np, rpm_test_labels_np),
    ("SimCLR", simclr_test_feats_np, simclr_test_labels_np),
]):
    pca2 = PCA(n_components=2)
    z2 = pca2.fit_transform(feats)
    plot_2d_scatter(z2, labels, f"PCA 2D - {name}", CIFAR10_CLASSES, axes[0, col])

# --- t-SNE ---
for col, (name, feats, labels) in enumerate([
    ("RPM", rpm_test_feats_np, rpm_test_labels_np),
    ("SimCLR", simclr_test_feats_np, simclr_test_labels_np),
]):
    idx = np.random.choice(len(feats), min(N_VIS, len(feats)), replace=False)
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca",
                learning_rate="auto")
    z2 = tsne.fit_transform(feats[idx])
    plot_2d_scatter(z2, labels[idx], f"t-SNE - {name}", CIFAR10_CLASSES,
                    axes[1, col])

handles, leg_labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, leg_labels, loc="lower center", ncol=5, fontsize=9,
           markerscale=3)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.suptitle("Dimensionality Reduction", fontsize=14, fontweight="bold", y=1.01)
plt.show()

# --- PCA explained variance curve ---
fig, ax = plt.subplots(figsize=(8, 4))
for name, feats in [("RPM", rpm_test_feats_np),
                     ("SimCLR", simclr_test_feats_np)]:
    pca_full = PCA().fit(feats)
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    ax.plot(cumvar, label=name)
    n90 = np.searchsorted(cumvar, 0.90) + 1
    n95 = np.searchsorted(cumvar, 0.95) + 1
    print(f"{name}: dims for 90% var = {n90}, 95% var = {n95}, "
          f"total dims = {feats.shape[1]}")
ax.axhline(0.90, ls="--", color="gray", alpha=0.5, label="90%")
ax.axhline(0.95, ls="--", color="gray", alpha=0.3, label="95%")
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Cumulative explained variance")
ax.set_title("Effective Dimensionality")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# =====================================================================
# 3. k-NN Accuracy
# =====================================================================
print("k-NN accuracy (test set):")
print(f"{'k':>4s}  {'RPM':>8s}  {'SimCLR':>8s}")
print("-" * 26)
for k in [1, 5, 10, 20]:
    knn_rpm = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    knn_rpm.fit(rpm_train_feats_np, rpm_train_labels_np)
    rpm_knn_acc = knn_rpm.score(rpm_test_feats_np, rpm_test_labels_np)

    knn_simclr = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    knn_simclr.fit(simclr_train_feats_np, simclr_train_labels_np)
    simclr_knn_acc = knn_simclr.score(simclr_test_feats_np, simclr_test_labels_np)

    print(f"{k:4d}  {rpm_knn_acc*100:7.2f}%  {simclr_knn_acc*100:7.2f}%")


In [ ]:
# =====================================================================
# 4. Alignment & Uniformity (Wang & Isola, 2020)
# =====================================================================

def compute_alignment(feats, labels, alpha=2):
    """Mean pairwise distance between same-class representations (L2, l2-normalized)."""
    feats_norm = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-8)
    total = 0.0
    count = 0
    classes = np.unique(labels)
    for c in classes:
        z_c = feats_norm[labels == c]
        n = len(z_c)
        if n < 2:
            continue
        # Pairwise distances within class
        dists = pdist(z_c, metric="sqeuclidean")
        total += np.sum(dists ** (alpha / 2))
        count += len(dists)
    return total / max(count, 1)


def compute_uniformity(feats, t=2, max_samples=5000):
    """Log of expected pairwise Gaussian kernel (l2-normalized features)."""
    feats_norm = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-8)
    if len(feats_norm) > max_samples:
        idx = np.random.choice(len(feats_norm), max_samples, replace=False)
        feats_norm = feats_norm[idx]
    sq_dists = pdist(feats_norm, metric="sqeuclidean")
    return np.log(np.mean(np.exp(-t * sq_dists)))


print("Alignment & Uniformity (lower is better for both):")
print(f"{'Metric':<15s}  {'RPM':>10s}  {'SimCLR':>10s}")
print("-" * 40)
for name, feats, labels in [
    ("RPM", rpm_test_feats_np, rpm_test_labels_np),
    ("SimCLR", simclr_test_feats_np, simclr_test_labels_np),
]:
    align = compute_alignment(feats, labels)
    uniform = compute_uniformity(feats)
    print(f"  {name:<13s}  align={align:.4f}  uniform={uniform:.4f}")


In [ ]:
# =====================================================================
# 5. Inter-class / Intra-class Structure
# =====================================================================

def class_centroids_and_variance(feats, labels, class_names):
    n_classes = len(class_names)
    centroids = np.zeros((n_classes, feats.shape[1]))
    intra_var = np.zeros(n_classes)
    for c in range(n_classes):
        z_c = feats[labels == c]
        centroids[c] = z_c.mean(axis=0)
        intra_var[c] = np.mean(np.sum((z_c - centroids[c]) ** 2, axis=1))
    return centroids, intra_var

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, feats, labels) in enumerate([
    ("RPM", rpm_test_feats_np, rpm_test_labels_np),
    ("SimCLR", simclr_test_feats_np, simclr_test_labels_np),
]):
    centroids, intra_var = class_centroids_and_variance(
        feats, labels, CIFAR10_CLASSES)
    # Cosine distance between centroids
    centroid_norm = centroids / (np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-8)
    cos_sim = centroid_norm @ centroid_norm.T
    cos_dist = 1 - cos_sim

    im = axes[idx].imshow(cos_dist, cmap="viridis", vmin=0)
    axes[idx].set_xticks(range(10))
    axes[idx].set_xticklabels(CIFAR10_CLASSES, rotation=45, ha="right", fontsize=8)
    axes[idx].set_yticks(range(10))
    axes[idx].set_yticklabels(CIFAR10_CLASSES, fontsize=8)
    axes[idx].set_title(f"Centroid cosine distance - {name}")
    plt.colorbar(im, ax=axes[idx], shrink=0.8)

# Intra-class variance comparison
rpm_centroids, rpm_intra = class_centroids_and_variance(
    rpm_test_feats_np, rpm_test_labels_np, CIFAR10_CLASSES)
simclr_centroids, simclr_intra = class_centroids_and_variance(
    simclr_test_feats_np, simclr_test_labels_np, CIFAR10_CLASSES)

x = np.arange(10)
w = 0.35
axes[2].bar(x - w/2, rpm_intra, w, label="RPM", alpha=0.8)
axes[2].bar(x + w/2, simclr_intra, w, label="SimCLR", alpha=0.8)
axes[2].set_xticks(x)
axes[2].set_xticklabels(CIFAR10_CLASSES, rotation=45, ha="right", fontsize=8)
axes[2].set_ylabel("Mean intra-class L2 variance")
axes[2].set_title("Intra-class spread")
axes[2].legend()

plt.suptitle("Inter-class / Intra-class Structure", fontsize=14,
              fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# =====================================================================
# 6. RPM-Specific: Uncertainty Analysis
# =====================================================================
# RPM outputs a Gaussian posterior; extract precision (inverse covariance).

@jax.jit
def rpm_posterior_params(params, views):
    rpm_output, _ = gaussian_model.apply({"params": params}, views)
    variational = rpm_output["variational"]
    variational_mean = rpm_output["variational_mean"]
    # For diag precision: uncertainty ~ 1/precision_diag
    return variational_mean.mean, variational.precision_diag

# Collect posterior means and precisions
rpm_means_list, rpm_precs_list, rpm_ulabels_list = [], [], []
for _, views, labels in rpm_eval_test_cifar10:
    mean, prec = rpm_posterior_params(rpm_final_state.params, views)
    rpm_means_list.append(np.array(mean))
    rpm_precs_list.append(np.array(prec))
    rpm_ulabels_list.append(np.array(labels))

rpm_means_all = np.concatenate(rpm_means_list, axis=0)
rpm_precs_all = np.concatenate(rpm_precs_list, axis=0)
rpm_ulabels_all = np.concatenate(rpm_ulabels_list, axis=0)

# Uncertainty = trace(covariance) = sum(1/precision_diag)
rpm_uncertainty = np.sum(1.0 / (rpm_precs_all + 1e-8), axis=-1)

# --- Per-class uncertainty ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_uncertainties = []
for c in range(10):
    mask = rpm_ulabels_all == c
    class_uncertainties.append(rpm_uncertainty[mask])

axes[0].boxplot(class_uncertainties, labels=CIFAR10_CLASSES, vert=True)
axes[0].set_xticklabels(CIFAR10_CLASSES, rotation=45, ha="right", fontsize=9)
axes[0].set_ylabel("Uncertainty (trace of covariance)")
axes[0].set_title("RPM: Per-class uncertainty distribution")

# --- Uncertainty calibration: kNN-based correctness vs uncertainty ---
knn_rpm_cal = KNeighborsClassifier(n_neighbors=5, metric="cosine")
knn_rpm_cal.fit(rpm_train_feats_np, rpm_train_labels_np)
rpm_knn_preds = knn_rpm_cal.predict(rpm_test_feats_np)
rpm_is_correct = (rpm_knn_preds == rpm_test_labels_np)

n_bins = 10
sorted_idx = np.argsort(rpm_uncertainty)
bin_size = len(sorted_idx) // n_bins
bin_accs, bin_uncs = [], []
for b in range(n_bins):
    start = b * bin_size
    end = start + bin_size if b < n_bins - 1 else len(sorted_idx)
    idx = sorted_idx[start:end]
    bin_accs.append(np.mean(rpm_is_correct[idx]))
    bin_uncs.append(np.mean(rpm_uncertainty[idx]))

axes[1].bar(range(n_bins), bin_accs, alpha=0.7, label="Accuracy")
ax2 = axes[1].twinx()
ax2.plot(range(n_bins), bin_uncs, "r-o", label="Mean uncertainty")
axes[1].set_xlabel("Uncertainty bin (low -> high)")
axes[1].set_ylabel("k-NN Accuracy")
ax2.set_ylabel("Mean uncertainty", color="r")
axes[1].set_title("RPM: Uncertainty calibration")
axes[1].legend(loc="upper left")
ax2.legend(loc="upper right")

plt.suptitle("RPM Uncertainty Analysis", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

correct_unc = np.mean(rpm_uncertainty[rpm_is_correct])
incorrect_unc = np.mean(rpm_uncertainty[~rpm_is_correct])
print(f"Mean uncertainty (correct):   {correct_unc:.4f}")
print(f"Mean uncertainty (incorrect): {incorrect_unc:.4f}")
print(f"Ratio (incorrect/correct):    {incorrect_unc/correct_unc:.2f}x")


In [ ]:
# =====================================================================
# 7. CKA (Centered Kernel Alignment)
# =====================================================================

def linear_CKA(X, Y):
    """Linear CKA between two feature matrices (n_samples, n_features)."""
    X = X - X.mean(axis=0)
    Y = Y - Y.mean(axis=0)
    hsic_xy = np.linalg.norm(X.T @ Y, ord="fro") ** 2
    hsic_xx = np.linalg.norm(X.T @ X, ord="fro") ** 2
    hsic_yy = np.linalg.norm(Y.T @ Y, ord="fro") ** 2
    return hsic_xy / (np.sqrt(hsic_xx * hsic_yy) + 1e-10)

# Ensure same number of samples (use test set, same ordering from same dataset)
n = min(len(rpm_test_feats_np), len(simclr_test_feats_np))
cka_score = linear_CKA(rpm_test_feats_np[:n], simclr_test_feats_np[:n])
print(f"Linear CKA between RPM and SimCLR ({ANALYSIS_FEATURE_SOURCE} features): {cka_score:.4f}")
print(f"  (1.0 = identical geometry, 0.0 = completely different)")

# Also compute CKA within each method between train and test for sanity
n_tt = min(len(rpm_train_feats_np), len(rpm_test_feats_np))
rpm_self_cka = linear_CKA(rpm_train_feats_np[:n_tt], rpm_test_feats_np[:n_tt])
simclr_self_cka = linear_CKA(simclr_train_feats_np[:n_tt], simclr_test_feats_np[:n_tt])
print(f"RPM    train-test CKA (self-consistency): {rpm_self_cka:.4f}")
print(f"SimCLR train-test CKA (self-consistency): {simclr_self_cka:.4f}")


---
# Experiment 3 -- Shapes3D disentanglement

Train RPM and SimCLR on the **Shapes3D** dataset (480k synthetic 64x64
images with 6 independent ground-truth factors: floor hue, wall hue,
object hue, scale, shape, orientation).

The goal is **not** classification -- instead we analyse whether the
learned latent dimensions capture the underlying generative factors.


In [ ]:
# =====================================================================
# Shapes3D data loader
# =====================================================================
SHAPES3D_FACTOR_NAMES = [
    "floor_hue", "wall_hue", "object_hue", "scale", "shape", "orientation",
]

SHAPES3D_MEAN = np.asarray([0.5, 0.5, 0.5], dtype=np.float32)
SHAPES3D_STD  = np.asarray([0.5, 0.5, 0.5], dtype=np.float32)


def shapes3d_simclr_augment(image, image_size=64, pass_mask=False):
    image = _random_resized_crop(image, size=image_size)
    image = tf.image.random_flip_left_right(image)
    do_jitter = tf.random.uniform([]) < 0.8
    image = tf.cond(do_jitter, lambda: _color_jitter(image, 0.5), lambda: image)
    image = _random_grayscale(image, p=0.2)
    do_blur = tf.random.uniform([]) < 0.5
    image = tf.cond(do_blur, lambda: _gaussian_blur(image, kernel_size=5),
                    lambda: image)
    image = (image - SHAPES3D_MEAN) / SHAPES3D_STD
    if pass_mask:
        image = _append_ones_mask(image)
    return image


def shapes3d_masking_augment(image, image_size=64, num_blocks=4,
                              scale_range=(0.15, 0.2),
                              aspect_ratio_range=(0.75, 1.5),
                              pass_mask=False):
    if image_size != 64:
        image = tf.image.resize(image, [image_size, image_size])
    image = (image - SHAPES3D_MEAN) / SHAPES3D_STD
    mask = _sample_multi_block_mask(image_size, image_size,
                                    num_blocks=num_blocks,
                                    scale_range=scale_range,
                                    aspect_ratio_range=aspect_ratio_range)
    mask_3d = tf.cast(mask[:, :, tf.newaxis], tf.float32)
    image = image * (1.0 - mask_3d)
    if pass_mask:
        visibility = 1.0 - mask_3d
        image = tf.concat([image, visibility], axis=-1)
    return image


def shapes3d_eval_preprocess(image, image_size=64, pass_mask=False):
    if image_size != 64:
        image = tf.image.resize(image, [image_size, image_size])
    image = (image - SHAPES3D_MEAN) / SHAPES3D_STD
    if pass_mask:
        image = _append_ones_mask(image)
    return image


class Shapes3DLoader:
    """Shapes3D data loader.

    Yields (raw_images, views, label) where label is label_shape (used as
    a dummy for the training loop). Ground-truth factor values are
    available separately via load_shapes3d_factors().

    When ``pass_mask=True``, each view has shape [H, W, 4] where the 4th
    channel is a visibility mask (1=visible, 0=masked).
    """

    def __init__(self, *, split_spec, batch_size, training, num_views=4,
                 image_size=64, preprocess_type="simclr", seed=None,
                 drop_remainder=None, cache=True, pass_mask=False):
        self._batch_size = batch_size
        self._training = training
        self._num_views = num_views
        self._image_size = image_size
        self._preprocess_type = preprocess_type
        self._pass_mask = pass_mask
        self._num_classes = 4   # shape factor has 4 categories
        if drop_remainder is None:
            drop_remainder = training
        self._drop_remainder = drop_remainder

        ds = tfds.load("shapes3d", split=split_spec, shuffle_files=training)
        self._dataset_size = ds.cardinality().numpy()
        if self._dataset_size < 0:
            # cardinality unknown; shapes3d has 480000 total
            self._dataset_size = 480000

        if cache:
            ds = ds.cache()
        if training:
            ds = ds.shuffle(buffer_size=min(self._dataset_size, 50_000),
                            seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(self._preprocess, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=drop_remainder)
        ds = ds.prefetch(tf.data.AUTOTUNE)
        self._ds = ds

    @property
    def dataset_size(self):
        return self._dataset_size

    @property
    def num_classes(self):
        return self._num_classes

    def _preprocess(self, example):
        image = example["image"]
        label = example["label_shape"]   # 0-3, used as dummy label
        raw_image = image
        image_f = tf.cast(image, tf.float32) / 255.0

        if self._training:
            if self._preprocess_type == "simclr":
                views = tf.stack([
                    shapes3d_simclr_augment(image_f, self._image_size,
                                            pass_mask=self._pass_mask)
                    for _ in range(self._num_views)
                ], axis=0)
            elif self._preprocess_type == "lejepa_masking":
                views = tf.stack([
                    shapes3d_masking_augment(image_f, self._image_size,
                                             pass_mask=self._pass_mask)
                    for _ in range(self._num_views)
                ], axis=0)
            else:
                raise ValueError(f"Unknown preprocess_type: {self._preprocess_type}")
        else:
            v = shapes3d_eval_preprocess(image_f, self._image_size,
                                         pass_mask=self._pass_mask)
            views = tf.stack([v] * self._num_views, axis=0)

        return raw_image, views, label

    def __iter__(self):
        for raw, views, labels in self._ds:
            yield (raw.numpy(), views.numpy(), labels.numpy())

    def __len__(self):
        if self._drop_remainder:
            return self._dataset_size // self._batch_size
        return math.ceil(self._dataset_size / self._batch_size)


def load_shapes3d_factors(split_spec, max_samples=None):
    """Load ground-truth factor values for analysis."""
    ds = tfds.load("shapes3d", split=split_spec)
    factors = {n: [] for n in SHAPES3D_FACTOR_NAMES}
    for ex in ds:
        for n in SHAPES3D_FACTOR_NAMES:
            factors[n].append(float(ex[f"value_{n}"].numpy()))
        if max_samples and len(factors["shape"]) >= max_samples:
            break
    return {n: np.array(v) for n, v in factors.items()}


print("Shapes3D loader defined.")


In [ ]:
# --- Shapes3D experiment config ---
S3D_SEED = 42
S3D_BATCH_SIZE = 128
S3D_EVAL_BATCH_SIZE = 256
S3D_NUM_VIEWS = 8
S3D_IMAGE_SIZE = 64
S3D_LATENT_DIM = 128
S3D_PREPROCESS_TYPE = "simclr"   # "simclr" or "lejepa_masking"
# S3D_PREPROCESS_TYPE = "lejepa_masking"   # "simclr" or "lejepa_masking"
S3D_PASS_MASK = True            # if True, append visibility mask as 4th channel
S3D_NUM_STEPS = 100_000
S3D_PROBE_EVERY = 20_000
S3D_VAL_EVERY = 10000
S3D_LOG_EVERY = 10000
S3D_VAL_STEPS = 10
S3D_CHECKPOINT_EVERY = 20_000
S3D_LR = 1e-4
S3D_WEIGHT_DECAY = 1e-6
S3D_GRAD_CLIP = 1.0
S3D_PROBE_EPOCHS = 10
S3D_PROBE_LR = 0.1
S3D_PROJECTION_FEATURES = (2048,)
S3D_PRECISION_TYPE = "diag"
S3D_TEMPERATURE = 0.5

# Use 90% train, 10% eval
S3D_TRAIN_SPLIT = "train[:90%]"
S3D_EVAL_SPLIT = "train[90%:]"

set_global_seed(S3D_SEED)
print("Shapes3D config set.")



rpm_beta_schedule = optax.linear_schedule(
        init_value=0.1, end_value=1.0,
        transition_steps=5000, transition_begin=0,
    )


In [ ]:
# --- Shapes3D data loaders ---
s3d_train_loader = Shapes3DLoader(
    split_spec=S3D_TRAIN_SPLIT, batch_size=S3D_BATCH_SIZE,
    training=True, num_views=S3D_NUM_VIEWS, image_size=S3D_IMAGE_SIZE,
    preprocess_type=S3D_PREPROCESS_TYPE, seed=S3D_SEED,
    pass_mask=S3D_PASS_MASK,
)
s3d_eval_loader = Shapes3DLoader(
    split_spec=S3D_EVAL_SPLIT, batch_size=S3D_EVAL_BATCH_SIZE,
    training=False, num_views=S3D_NUM_VIEWS, image_size=S3D_IMAGE_SIZE,
    preprocess_type=S3D_PREPROCESS_TYPE, seed=S3D_SEED,
    pass_mask=S3D_PASS_MASK,
)

_b = next(iter(s3d_train_loader))
print("Shapes3D train batch shapes:", _b[0].shape, _b[1].shape, _b[2].shape)
print(f"Train size: {s3d_train_loader.dataset_size}, "
      f"Eval size: {s3d_eval_loader.dataset_size}")


In [ ]:
# --- ResNet-18 config adjusted for 64x64 images ---
def get_resnet18_cfg_64(projection_features):
    return {
        "stage_sizes": (2, 2, 2, 2),
        "stage_widths": (64, 128, 256, 512),
        "stem_width": 64,
        "stem_kernel_size": (3, 3),
        "stem_stride": 1,
        "use_max_pool": True,
        "max_pool_window": (3, 3),
        "max_pool_stride": (2, 2),
        "block_type": "basic",
        "projection_features": projection_features,
    }

# --- RPM model ---
s3d_rpm_model = GaussianRPM.create(
    auxiliary_method="constrained",
    n_factors=S3D_NUM_VIEWS,
    dim_latent=S3D_LATENT_DIM,
    encoder_arch="resnet",
    encoder_params=get_resnet18_cfg_64(S3D_PROJECTION_FEATURES),
    share_recognition=True,
    precision_type=S3D_PRECISION_TYPE,
    fix_prior=True,
)
print("Shapes3D RPM model constructed.")

# --- SimCLR model ---
s3d_simclr_model = SimCLREncoder(
    n_views=S3D_NUM_VIEWS,
    encoder_arch="resnet",
    encoder_params=get_resnet18_cfg_64(S3D_PROJECTION_FEATURES),
    temperature=S3D_TEMPERATURE,
)
print("Shapes3D SimCLR model constructed.")


In [ ]:
# --- Train RPM on Shapes3D ---
s3d_rpm_output_dir = Path("/tmp/outputs/shapes3d_rpm") / time.strftime("%Y%m%d-%H%M%S")

s3d_rpm_state, s3d_rpm_history = train_rpm(
    model=s3d_rpm_model,
    train_dataloader=s3d_train_loader,
    eval_train_dataloader=s3d_eval_loader,
    eval_test_dataloader=s3d_eval_loader,
    optimizer_name="adamw",
    probe_type="linear",
    output_dir=s3d_rpm_output_dir,
    learning_rate=optax.constant_schedule(S3D_LR),
    weight_decay=S3D_WEIGHT_DECAY,
    grad_clip_norm=S3D_GRAD_CLIP,
    batch_size=S3D_BATCH_SIZE,
    num_steps=S3D_NUM_STEPS,
    log_every=S3D_LOG_EVERY,
    val_every=S3D_VAL_EVERY,
    val_steps=S3D_VAL_STEPS,
    probe_every=S3D_PROBE_EVERY,
    probe_epochs=S3D_PROBE_EPOCHS,
    probe_lr=S3D_PROBE_LR,
    seed=S3D_SEED,
    # beta=optax.constant_schedule(1.0),
    beta=rpm_beta_schedule,
    probe_feature_source="trunk",
    checkpoint_every=S3D_CHECKPOINT_EVERY,
    save_best=True,
    max_checkpoints_to_keep=3,
)
print("Shapes3D RPM training done.")


In [ ]:
print('da')

In [ ]:
s3d_rpm_history.keys()

In [ ]:
plt.plot(s3d_rpm_history['val_free_energy'])

In [ ]:
# --- Train SimCLR on Shapes3D ---
s3d_simclr_output_dir = Path("/tmp/outputs/shapes3d_simclr") / time.strftime("%Y%m%d-%H%M%S")

s3d_simclr_state, s3d_simclr_history = train_rpm(
    model=s3d_simclr_model,
    train_dataloader=s3d_train_loader,
    eval_train_dataloader=s3d_eval_loader,
    eval_test_dataloader=s3d_eval_loader,
    optimizer_name="adamw",
    probe_type="linear",
    output_dir=s3d_simclr_output_dir,
    learning_rate=optax.constant_schedule(S3D_LR),
    weight_decay=S3D_WEIGHT_DECAY,
    grad_clip_norm=S3D_GRAD_CLIP,
    batch_size=S3D_BATCH_SIZE,
    num_steps=S3D_NUM_STEPS,
    log_every=S3D_LOG_EVERY,
    val_every=S3D_VAL_EVERY,
    val_steps=S3D_VAL_STEPS,
    probe_every=S3D_PROBE_EVERY,
    probe_epochs=S3D_PROBE_EPOCHS,
    probe_lr=S3D_PROBE_LR,
    seed=S3D_SEED,
    probe_feature_source="trunk",
    checkpoint_every=S3D_CHECKPOINT_EVERY,
    save_best=True,
    max_checkpoints_to_keep=3,
)
print("Shapes3D SimCLR training done.")


In [ ]:
plt.plot(s3d_simclr_history['loss'])

In [ ]:
# =====================================================================
# Extract features and ground-truth factors for analysis
# =====================================================================

# --- RPM latent: variational posterior mean (aggregated across views) ---
@jax.jit
def _rpm_latent_fn(params, views):
    out, _ = s3d_rpm_model.apply({"params": params}, views)
    return out["variational_mean"].mean   # (batch, latent_dim)

s3d_rpm_latent_list, s3d_rpm_labels_list = [], []
for _, views, labels in s3d_eval_loader:
    feats = _rpm_latent_fn(s3d_rpm_state.params, views)
    s3d_rpm_latent_list.append(np.array(feats))
    s3d_rpm_labels_list.append(np.array(labels))

s3d_rpm_latent_np = np.concatenate(s3d_rpm_latent_list, axis=0)
s3d_rpm_labels_np = np.concatenate(s3d_rpm_labels_list, axis=0)

# --- RPM trunk: backbone features before projection head ---
s3d_rpm_trunk_fn = create_feature_extractor(s3d_rpm_model, feature_source="trunk")
s3d_rpm_trunk_feats, _ = compute_features_and_labels(
    s3d_rpm_trunk_fn, s3d_rpm_state.params, s3d_eval_loader)
s3d_rpm_trunk_np = np.array(s3d_rpm_trunk_feats)

# --- SimCLR trunk: backbone features before projection head ---
s3d_simclr_trunk_fn = create_feature_extractor(s3d_simclr_model,
                                                feature_source="trunk")
s3d_simclr_feats, s3d_simclr_labels = compute_features_and_labels(
    s3d_simclr_trunk_fn, s3d_simclr_state.params, s3d_eval_loader)
s3d_simclr_trunk_np = np.array(s3d_simclr_feats)
s3d_simclr_labels_np = np.array(s3d_simclr_labels)

for arr, label in [(s3d_rpm_latent_np, "RPM latent"),
                    (s3d_rpm_trunk_np, "RPM trunk"),
                    (s3d_simclr_trunk_np, "SimCLR trunk")]:
    assert arr.ndim == 2, f"Expected 2D for {label}, got shape {arr.shape}"

# Load ground-truth factor values (same split/ordering as eval loader)
s3d_factors = load_shapes3d_factors(S3D_EVAL_SPLIT,
                                    max_samples=len(s3d_rpm_latent_np))

# Convenience dict for iterating over all three representations
S3D_REPR = {
    "RPM latent": s3d_rpm_latent_np,
    "RPM trunk":  s3d_rpm_trunk_np,
    "SimCLR trunk": s3d_simclr_trunk_np,
}

print("Feature shapes:")
for name, arr in S3D_REPR.items():
    print(f"  {name}: {arr.shape}")
for fn in SHAPES3D_FACTOR_NAMES:
    print(f"  {fn}: {len(s3d_factors[fn])} values, "
          f"range [{s3d_factors[fn].min():.3f}, {s3d_factors[fn].max():.3f}]")


In [ ]:
# =====================================================================
# Disentanglement: R^2 of each latent dim vs each factor
# =====================================================================
from sklearn.linear_model import LinearRegression

def compute_r2_matrix(feats, factors, factor_names):
    """R^2 between each latent dimension and each ground-truth factor."""
    n_dims = feats.shape[1]
    n_factors = len(factor_names)
    r2 = np.zeros((n_factors, n_dims))
    for fi, fn in enumerate(factor_names):
        y = factors[fn]
        for di in range(n_dims):
            x = feats[:, di:di+1]
            reg = LinearRegression().fit(x, y)
            r2[fi, di] = max(0, reg.score(x, y))
    return r2

def compute_factor_r2(feats, factors, factor_names):
    """Per-factor R^2 using ALL latent dims (full linear model)."""
    results = {}
    for fn in factor_names:
        y = factors[fn]
        reg = LinearRegression().fit(feats, y)
        results[fn] = max(0, reg.score(feats, y))
    return results

# Compute R^2 for all three representations
s3d_r2_mats = {}
s3d_full_r2 = {}
for name, feats in S3D_REPR.items():
    print(f"Computing R^2 for {name} ({feats.shape[1]} dims)...")
    s3d_r2_mats[name] = compute_r2_matrix(feats, s3d_factors, SHAPES3D_FACTOR_NAMES)
    s3d_full_r2[name] = compute_factor_r2(feats, s3d_factors, SHAPES3D_FACTOR_NAMES)

# --- Heatmap: per-dim R^2 (top 20 dims) ---
n_repr = len(S3D_REPR)
fig, axes = plt.subplots(1, n_repr, figsize=(7 * n_repr, 4))
for ax, (name, r2_mat) in zip(axes, s3d_r2_mats.items()):
    max_r2_per_dim = r2_mat.max(axis=0)
    top_dims = np.argsort(max_r2_per_dim)[-20:]
    im = ax.imshow(r2_mat[:, top_dims], aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_yticks(range(len(SHAPES3D_FACTOR_NAMES)))
    ax.set_yticklabels(SHAPES3D_FACTOR_NAMES, fontsize=9)
    ax.set_xlabel("Latent dimension (top 20 by max R²)")
    ax.set_title(f"{name}: per-dim R²")
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

# --- Bar chart: full-model R^2 per factor ---
fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(SHAPES3D_FACTOR_NAMES))
w = 0.25
repr_names = list(S3D_REPR.keys())
colors = ["#1f77b4", "#2ca02c", "#ff7f0e"]
for i, name in enumerate(repr_names):
    vals = [s3d_full_r2[name][f] for f in SHAPES3D_FACTOR_NAMES]
    ax.bar(x + (i - 1) * w, vals, w, label=name, alpha=0.8, color=colors[i])
ax.set_xticks(x)
ax.set_xticklabels(SHAPES3D_FACTOR_NAMES, rotation=30, ha="right")
ax.set_ylabel("R² (linear regression)")
ax.set_title("Factor predictability from full representation")
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

print("\nFull-model R² per factor:")
header = f"{'Factor':<15s}" + "".join(f"  {n:>14s}" for n in repr_names)
print(header)
print("-" * len(header))
for fn in SHAPES3D_FACTOR_NAMES:
    row = f"{fn:<15s}" + "".join(f"  {s3d_full_r2[n][fn]:14.4f}" for n in repr_names)
    print(row)


In [ ]:
# =====================================================================
# t-SNE colored by each ground-truth factor
# =====================================================================
N_TSNE = 5000  # subsample for speed

idx = np.random.choice(len(s3d_rpm_latent_np),
                        min(N_TSNE, len(s3d_rpm_latent_np)), replace=False)

# Compute t-SNE for all three representations
s3d_tsne = {}
for name, feats in S3D_REPR.items():
    print(f"Running t-SNE for {name}...")
    s3d_tsne[name] = TSNE(n_components=2, perplexity=30, random_state=42,
                           init="pca", learning_rate="auto"
                           ).fit_transform(feats[idx])

n_factors = len(SHAPES3D_FACTOR_NAMES)
n_repr = len(S3D_REPR)
fig, axes = plt.subplots(n_repr, n_factors, figsize=(4 * n_factors, 4 * n_repr))

repr_names = list(S3D_REPR.keys())
for row, name in enumerate(repr_names):
    z2 = s3d_tsne[name]
    for fi, fn in enumerate(SHAPES3D_FACTOR_NAMES):
        fv = s3d_factors[fn][idx]
        ax = axes[row, fi]
        sc = ax.scatter(z2[:, 0], z2[:, 1], c=fv, s=2, alpha=0.5, cmap="viridis")
        ax.set_title(f"{name}: {fn}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        if fi == n_factors - 1:
            plt.colorbar(sc, ax=ax, shrink=0.7)

plt.suptitle("t-SNE colored by ground-truth factors", fontsize=14,
              fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# =====================================================================
# PCA explained variance + factor-latent correlation matrix
# =====================================================================

# --- Explained variance ---
fig, ax = plt.subplots(figsize=(8, 4))
for name, feats in S3D_REPR.items():
    pca = PCA().fit(feats)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    ax.plot(cumvar, label=name)
    n90 = np.searchsorted(cumvar, 0.90) + 1
    n95 = np.searchsorted(cumvar, 0.95) + 1
    print(f"{name}: dims for 90% var = {n90}, 95% var = {n95}")
ax.axhline(0.90, ls="--", color="gray", alpha=0.5)
ax.axhline(0.95, ls="--", color="gray", alpha=0.3)
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Cumulative explained variance")
ax.set_title("Shapes3D: Effective dimensionality")
ax.legend()
plt.tight_layout()
plt.show()

# --- Correlation matrix: PCA components vs factors ---
n_repr = len(S3D_REPR)
fig, axes = plt.subplots(1, n_repr, figsize=(7 * n_repr, 5))
for ax, (name, feats) in zip(axes, S3D_REPR.items()):
    pca = PCA(n_components=min(20, feats.shape[1])).fit_transform(feats)
    corr = np.zeros((len(SHAPES3D_FACTOR_NAMES), pca.shape[1]))
    for fi, fn in enumerate(SHAPES3D_FACTOR_NAMES):
        fv = s3d_factors[fn][:len(pca)]
        for pc in range(pca.shape[1]):
            corr[fi, pc] = np.abs(np.corrcoef(fv, pca[:, pc])[0, 1])
    im = ax.imshow(corr, aspect="auto", cmap="Blues", vmin=0, vmax=1)
    ax.set_yticks(range(len(SHAPES3D_FACTOR_NAMES)))
    ax.set_yticklabels(SHAPES3D_FACTOR_NAMES, fontsize=9)
    ax.set_xlabel("PCA component")
    ax.set_title(f"{name}: |corr| of PCA dims vs factors")
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()
